# Pipeline End-to-End — Indicador Criança Alfabetizada

Este notebook orquestra a execução completa da pipeline batch do Tech Challenge — Fase 2.

## Objetivo

Executar as etapas na ordem correta:

1. Ingestão das fontes históricas na camada Bronze;
2. Validação da ingestão;
3. Transformação e integração na camada Silver;
4. Validação da camada Silver;
5. Construção das tabelas analíticas Gold;
6. Validação final da pipeline;
7. Geração de métricas e logs operacionais.

## Segurança

O notebook será iniciado em modo de preparação.

Enquanto `EXECUTE_PIPELINE` estiver definido como `False`, nenhuma tabela será criada ou substituída.

A execução real será liberada somente depois que todos os scripts finais forem consolidados e revisados.

## Escopo

A simulação de streaming permanece isolada e não participa da atualização das tabelas oficiais.

O fluxo deste notebook utiliza exclusivamente dados públicos reais da Base dos Dados.

In [1]:
from google.colab import auth
from google.cloud import bigquery

from datetime import datetime, timezone
from pathlib import Path

import json
import time
import uuid

import pandas as pd

auth.authenticate_user()

PROJECT_ID = "macro-coil-475920-k5"
LOCATION = "US"

SOURCE_PROJECT = "basedosdados"
SOURCE_DATASET = "br_inep_avaliacao_alfabetizacao"

BRONZE_DATASET = "bronze"
SILVER_DATASET = "silver"
GOLD_DATASET = "gold"

EXECUTE_PIPELINE = False

execution_id = str(uuid.uuid4())
started_at = datetime.now(timezone.utc)

client = bigquery.Client(
    project=PROJECT_ID,
    location=LOCATION,
)

print("=" * 70)
print("PIPELINE END-TO-END — AMBIENTE PREPARADO")
print("=" * 70)
print(f"Projeto: {PROJECT_ID}")
print(f"Localização: {LOCATION}")
print(f"Execution ID: {execution_id}")
print(f"Início UTC: {started_at.isoformat()}")
print(f"Execução liberada: {EXECUTE_PIPELINE}")

if not EXECUTE_PIPELINE:
    print(
        "\nModo seguro ativo. "
        "Nenhuma tabela será modificada nesta etapa."
    )

PIPELINE END-TO-END — AMBIENTE PREPARADO
Projeto: macro-coil-475920-k5
Localização: US
Execution ID: edd46773-51b2-4145-b489-09a398e33c9a
Início UTC: 2026-07-12T14:59:41.203759+00:00
Execução liberada: False

Modo seguro ativo. Nenhuma tabela será modificada nesta etapa.


## 2. Controle de execução e registro das etapas

Nesta etapa, será criada a função responsável por executar os scripts SQL do pipeline.

Cada etapa terá:

- nome da camada;
- nome da transformação;
- horário de início e término;
- duração;
- quantidade de bytes processados;
- status de sucesso, falha ou preparação;
- mensagem de erro, quando aplicável.

Enquanto o modo seguro estiver ativo, os scripts serão registrados como preparados, mas não serão enviados ao BigQuery.

Quando a execução for liberada, o fluxo seguirá esta regra:

1. Executar uma etapa;
2. Aguardar sua conclusão;
3. Registrar o resultado;
4. Interromper imediatamente em caso de falha;
5. Prosseguir somente quando a etapa anterior tiver sido concluída com sucesso.

In [3]:
pipeline_logs = []


def executar_sql(
    camada,
    etapa,
    sql,
    executar=False,
):
    inicio_etapa = datetime.now(timezone.utc)
    cronometro = time.perf_counter()

    if not executar:
        registro = {
            "execution_id": execution_id,
            "camada": camada,
            "etapa": etapa,
            "inicio_utc": inicio_etapa.isoformat(),
            "fim_utc": datetime.now(timezone.utc).isoformat(),
            "duracao_segundos": 0,
            "status": "PREPARADO",
            "bytes_processados": 0,
            "mensagem_erro": None,
        }

        pipeline_logs.append(registro)

        print(
            f"[PREPARADO] {camada} | {etapa}"
        )

        return registro

    try:
        job_config = bigquery.QueryJobConfig(
            labels={
                "pipeline": "alfabetizacao",
                "camada": camada.lower(),
                "tipo": "end_to_end",
            }
        )

        query_job = client.query(
            sql,
            location=LOCATION,
            job_config=job_config,
        )

        query_job.result()

        duracao = round(
            time.perf_counter() - cronometro,
            2,
        )

        registro = {
            "execution_id": execution_id,
            "camada": camada,
            "etapa": etapa,
            "inicio_utc": inicio_etapa.isoformat(),
            "fim_utc": datetime.now(timezone.utc).isoformat(),
            "duracao_segundos": duracao,
            "status": "SUCESSO",
            "bytes_processados": (
                query_job.total_bytes_processed or 0
            ),
            "mensagem_erro": None,
        }

        pipeline_logs.append(registro)

        print(
            f"[SUCESSO] {camada} | {etapa} | "
            f"{duracao}s"
        )

        return registro

    except Exception as erro:
        duracao = round(
            time.perf_counter() - cronometro,
            2,
        )

        registro = {
            "execution_id": execution_id,
            "camada": camada,
            "etapa": etapa,
            "inicio_utc": inicio_etapa.isoformat(),
            "fim_utc": datetime.now(timezone.utc).isoformat(),
            "duracao_segundos": duracao,
            "status": "ERRO",
            "bytes_processados": None,
            "mensagem_erro": str(erro),
        }

        pipeline_logs.append(registro)

        print(
            f"[ERRO] {camada} | {etapa}"
        )

        raise RuntimeError(
            f"Pipeline interrompido na etapa "
            f"{camada} | {etapa}: {erro}"
        ) from erro

## 3. Teste do mecanismo de execução

Antes de carregar os scripts reais, será realizado um teste em modo seguro.

A consulta utilizada será simples e não será enviada ao BigQuery porque a variável `EXECUTE_PIPELINE` continua definida como `False`.

O objetivo é confirmar que:

- a função recebe uma etapa;
- registra a camada e o nome;
- respeita o modo seguro;
- adiciona o resultado ao log;
- não altera nenhuma tabela.

In [4]:
sql_teste = """
SELECT
  1 AS teste_pipeline
"""

executar_sql(
    camada="CONTROLE",
    etapa="teste_modo_seguro",
    sql=sql_teste,
    executar=EXECUTE_PIPELINE,
)

pipeline_logs_df = pd.DataFrame(
    pipeline_logs
)

display(pipeline_logs_df)

[PREPARADO] CONTROLE | teste_modo_seguro


,execution_id,camada,etapa,inicio_utc,fim_utc,duracao_segundos,status,bytes_processados,mensagem_erro
0,edd46773-51b2-4145-b489-09a398e33c9a,CONTROLE,teste_modo_seguro,2026-07-12T15:05:53.909784+00:00,2026-07-12T15:05:53.909810+00:00,0,PREPARADO,0,None


## 4. Plano de execução da camada Bronze

A camada Bronze será atualizada por carga completa das sete entidades públicas da Base dos Dados.

Para cada entidade, o orquestrador preparará uma instrução `CREATE OR REPLACE TABLE`, responsável por substituir a tabela Bronze pela versão mais recente da fonte.

Entidades processadas:

- alunos;
- dicionário;
- metas nacionais;
- metas estaduais;
- metas municipais;
- indicadores municipais;
- indicadores por UF.

Como o modo seguro permanece ativo, as instruções serão apenas registradas como preparadas.

Nenhuma tabela será modificada nesta etapa.

In [5]:
bronze_tables = [
    "alunos",
    "dicionario",
    "meta_alfabetizacao_brasil",
    "meta_alfabetizacao_municipio",
    "meta_alfabetizacao_uf",
    "municipio",
    "uf",
]

bronze_steps = []

for table_name in bronze_tables:
    source_table = (
        f"{SOURCE_PROJECT}."
        f"{SOURCE_DATASET}."
        f"{table_name}"
    )

    destination_table = (
        f"{PROJECT_ID}."
        f"{BRONZE_DATASET}."
        f"{table_name}"
    )

    sql = f"""
    CREATE OR REPLACE TABLE
      `{destination_table}`

    AS

    SELECT
      *

    FROM
      `{source_table}`
    """

    bronze_steps.append(
        {
            "camada": "BRONZE",
            "etapa": f"carregar_{table_name}",
            "tabela_origem": source_table,
            "tabela_destino": destination_table,
            "sql": sql,
        }
    )

pipeline_logs.clear()

for step in bronze_steps:
    executar_sql(
        camada=step["camada"],
        etapa=step["etapa"],
        sql=step["sql"],
        executar=EXECUTE_PIPELINE,
    )

bronze_plan_df = pd.DataFrame(
    [
        {
            "camada": step["camada"],
            "etapa": step["etapa"],
            "tabela_origem": step["tabela_origem"],
            "tabela_destino": step["tabela_destino"],
        }
        for step in bronze_steps
    ]
)

pipeline_logs_df = pd.DataFrame(pipeline_logs)

display(bronze_plan_df)
display(pipeline_logs_df)

[PREPARADO] BRONZE | carregar_alunos
[PREPARADO] BRONZE | carregar_dicionario
[PREPARADO] BRONZE | carregar_meta_alfabetizacao_brasil
[PREPARADO] BRONZE | carregar_meta_alfabetizacao_municipio
[PREPARADO] BRONZE | carregar_meta_alfabetizacao_uf
[PREPARADO] BRONZE | carregar_municipio
[PREPARADO] BRONZE | carregar_uf


,camada,etapa,tabela_origem,tabela_destino
0,BRONZE,carregar_alunos,basedosdados.br_inep_avaliacao_alfabetizacao.a...,macro-coil-475920-k5.bronze.alunos
1,BRONZE,carregar_dicionario,basedosdados.br_inep_avaliacao_alfabetizacao.d...,macro-coil-475920-k5.bronze.dicionario
2,BRONZE,carregar_meta_alfabetizacao_brasil,basedosdados.br_inep_avaliacao_alfabetizacao.m...,macro-coil-475920-k5.bronze.meta_alfabetizacao...
3,BRONZE,carregar_meta_alfabetizacao_municipio,basedosdados.br_inep_avaliacao_alfabetizacao.m...,macro-coil-475920-k5.bronze.meta_alfabetizacao...
4,BRONZE,carregar_meta_alfabetizacao_uf,basedosdados.br_inep_avaliacao_alfabetizacao.m...,macro-coil-475920-k5.bronze.meta_alfabetizacao_uf
5,BRONZE,carregar_municipio,basedosdados.br_inep_avaliacao_alfabetizacao.m...,macro-coil-475920-k5.bronze.municipio
6,BRONZE,carregar_uf,basedosdados.br_inep_avaliacao_alfabetizacao.uf,macro-coil-475920-k5.bronze.uf


,execution_id,camada,etapa,inicio_utc,fim_utc,duracao_segundos,status,bytes_processados,mensagem_erro
0,edd46773-51b2-4145-b489-09a398e33c9a,BRONZE,carregar_alunos,2026-07-12T15:26:13.927152+00:00,2026-07-12T15:26:13.927169+00:00,0,PREPARADO,0,None
1,edd46773-51b2-4145-b489-09a398e33c9a,BRONZE,carregar_dicionario,2026-07-12T15:26:13.927232+00:00,2026-07-12T15:26:13.927241+00:00,0,PREPARADO,0,None
2,edd46773-51b2-4145-b489-09a398e33c9a,BRONZE,carregar_meta_alfabetizacao_brasil,2026-07-12T15:26:13.927259+00:00,2026-07-12T15:26:13.927266+00:00,0,PREPARADO,0,None
3,edd46773-51b2-4145-b489-09a398e33c9a,BRONZE,carregar_meta_alfabetizacao_municipio,2026-07-12T15:26:13.927281+00:00,2026-07-12T15:26:13.927287+00:00,0,PREPARADO,0,None
4,edd46773-51b2-4145-b489-09a398e33c9a,BRONZE,carregar_meta_alfabetizacao_uf,2026-07-12T15:26:13.927298+00:00,2026-07-12T15:26:13.927302+00:00,0,PREPARADO,0,None
5,edd46773-51b2-4145-b489-09a398e33c9a,BRONZE,carregar_municipio,2026-07-12T15:26:13.927311+00:00,2026-07-12T15:26:13.927315+00:00,0,PREPARADO,0,None
6,edd46773-51b2-4145-b489-09a398e33c9a,BRONZE,carregar_uf,2026-07-12T15:26:13.927328+00:00,2026-07-12T15:26:13.927332+00:00,0,PREPARADO,0,None


## 5. Quality gate da camada Bronze

Esta etapa verifica se a camada Bronze está íntegra antes de permitir o processamento da Silver.

Para cada entidade, serão validados:

- existência da tabela de origem;
- existência da tabela Bronze;
- quantidade de registros na origem;
- quantidade de registros na Bronze;
- correspondência entre as contagens;
- tamanho aproximado dos dados.

A validação utiliza os metadados do BigQuery e não modifica nenhuma tabela.

Quando o pipeline for executado de verdade, esse mesmo quality gate será aplicado imediatamente após a carga Bronze. Caso alguma tabela esteja ausente ou apresente divergência de volume, o fluxo será interrompido antes da Silver.

In [6]:
from google.api_core.exceptions import Forbidden, NotFound


def validar_bronze():
    resultados = []

    for table_name in bronze_tables:
        source_table_id = (
            f"{SOURCE_PROJECT}."
            f"{SOURCE_DATASET}."
            f"{table_name}"
        )

        destination_table_id = (
            f"{PROJECT_ID}."
            f"{BRONZE_DATASET}."
            f"{table_name}"
        )

        try:
            source_table = client.get_table(source_table_id)
            origem_existe = True
            linhas_origem = source_table.num_rows
            bytes_origem = source_table.num_bytes
        except (NotFound, Forbidden):
            origem_existe = False
            linhas_origem = None
            bytes_origem = None

        try:
            destination_table = client.get_table(
                destination_table_id
            )
            bronze_existe = True
            linhas_bronze = destination_table.num_rows
            bytes_bronze = destination_table.num_bytes
        except (NotFound, Forbidden):
            bronze_existe = False
            linhas_bronze = None
            bytes_bronze = None

        contagem_confere = (
            origem_existe
            and bronze_existe
            and linhas_origem == linhas_bronze
        )

        resultados.append(
            {
                "tabela": table_name,
                "origem_existe": origem_existe,
                "bronze_existe": bronze_existe,
                "linhas_origem": linhas_origem,
                "linhas_bronze": linhas_bronze,
                "contagem_confere": contagem_confere,
                "mb_origem": (
                    round(bytes_origem / 1024**2, 2)
                    if bytes_origem is not None
                    else None
                ),
                "mb_bronze": (
                    round(bytes_bronze / 1024**2, 2)
                    if bytes_bronze is not None
                    else None
                ),
            }
        )

    resultados_df = pd.DataFrame(resultados)

    quality_gate_aprovado = bool(
        resultados_df["origem_existe"].all()
        and resultados_df["bronze_existe"].all()
        and resultados_df["contagem_confere"].all()
        and len(resultados_df) == len(bronze_tables)
    )

    return resultados_df, quality_gate_aprovado


bronze_validation_df, bronze_quality_gate = validar_bronze()

display(bronze_validation_df)

print(
    "\nStatus do quality gate Bronze:",
    "APROVADO" if bronze_quality_gate else "REPROVADO",
)

if not bronze_quality_gate:
    raise RuntimeError(
        "Quality gate da Bronze reprovado. "
        "A Silver não deve ser processada."
    )

,tabela,origem_existe,bronze_existe,linhas_origem,linhas_bronze,contagem_confere,mb_origem,mb_bronze
0,alunos,True,True,3867999,3867999,True,256.10,256.10
1,dicionario,True,True,27,27,True,0.00,0.00
2,meta_alfabetizacao_brasil,True,True,3,3,True,0.00,0.00
3,meta_alfabetizacao_municipio,True,True,10704,10704,True,1.10,1.10
4,meta_alfabetizacao_uf,True,True,81,81,True,0.01,0.01
5,municipio,True,True,23995,23995,True,1.75,1.75
6,uf,True,True,145,145,True,0.01,0.01



Status do quality gate Bronze: APROVADO


## 6. Transformação da tabela Silver de alunos

Esta etapa prepara a transformação dos microdados de alunos da Bronze para a Silver.

A tabela Silver será responsável por:

- padronizar identificadores e campos textuais;
- converter códigos numéricos armazenados como texto;
- criar uma chave única por aluno e ano;
- identificar presença e preenchimento do caderno;
- classificar avaliações válidas e inválidas;
- preservar proficiência e peso ausentes sem substituí-los por zero;
- validar a classificação oficial com o corte de 743 pontos;
- preparar a tabela para análises e enriquecimentos posteriores.

A tabela será particionada por ano e clusterizada por município e rede.

Como o modo seguro está ativo, a transformação será apenas registrada como preparada.

In [8]:
sql_silver_alunos = f"""
CREATE OR REPLACE TABLE
  `{PROJECT_ID}.{SILVER_DATASET}.alunos`

PARTITION BY
  RANGE_BUCKET(ano, GENERATE_ARRAY(2023, 2025, 1))

CLUSTER BY
  id_municipio,
  rede

AS

WITH padronizacao AS (
  SELECT
    ano,
    TRIM(id_municipio) AS id_municipio,
    TRIM(id_escola) AS id_escola,
    TRIM(id_aluno) AS id_aluno,
    TRIM(caderno) AS caderno,
    TRIM(serie) AS serie,
    TRIM(rede) AS rede,
    SAFE_CAST(presenca AS INT64) AS presenca,
    SAFE_CAST(preenchimento_caderno AS INT64)
      AS preenchimento_caderno,
    SAFE_CAST(alfabetizado AS INT64)
      AS alfabetizado_oficial,
    proficiencia,
    peso_aluno

  FROM
    `{PROJECT_ID}.{BRONZE_DATASET}.alunos`
),

regras AS (
  SELECT
    CONCAT(
      CAST(ano AS STRING),
      '|',
      id_aluno
    ) AS chave_ano_aluno,

    ano,
    id_municipio,
    id_escola,
    id_aluno,
    caderno,
    serie,
    rede,
    presenca,
    preenchimento_caderno,
    alfabetizado_oficial,
    proficiencia,
    peso_aluno,

    presenca = 1 AS presente,

    preenchimento_caderno = 1
      AS caderno_preenchido,

    (
      presenca = 1
      AND preenchimento_caderno = 1
      AND proficiencia IS NOT NULL
      AND peso_aluno IS NOT NULL
    ) AS avaliacao_valida

  FROM
    padronizacao
)

SELECT
  chave_ano_aluno,
  ano,
  id_municipio,
  id_escola,
  id_aluno,
  caderno,
  serie,
  rede,
  presenca,
  preenchimento_caderno,
  alfabetizado_oficial,
  proficiencia,
  peso_aluno,
  presente,
  caderno_preenchido,
  avaliacao_valida,

  CASE
    WHEN presenca = 0
      THEN 'ausente'

    WHEN
      presenca = 1
      AND preenchimento_caderno = 0
      THEN 'presente_sem_preenchimento'

    WHEN avaliacao_valida
      THEN 'avaliacao_valida'

    ELSE 'situacao_inconsistente'
  END AS status_avaliacao,

  CASE
    WHEN avaliacao_valida
      THEN IF(proficiencia >= 743, 1, 0)

    ELSE NULL
  END AS alfabetizado_calculado_743,

  CASE
    WHEN avaliacao_valida
      THEN alfabetizado_oficial
        = IF(proficiencia >= 743, 1, 0)

    ELSE NULL
  END AS classificacao_coerente_743,

  CURRENT_TIMESTAMP() AS processado_em

FROM
  regras
"""

executar_sql(
    camada="SILVER",
    etapa="criar_silver_alunos",
    sql=sql_silver_alunos,
    executar=EXECUTE_PIPELINE,
)

display(
    pd.DataFrame(pipeline_logs).tail(1)
)

[PREPARADO] SILVER | criar_silver_alunos


,execution_id,camada,etapa,inicio_utc,fim_utc,duracao_segundos,status,bytes_processados,mensagem_erro
8,edd46773-51b2-4145-b489-09a398e33c9a,SILVER,criar_silver_alunos,2026-07-12T15:32:22.214260+00:00,2026-07-12T15:32:22.214283+00:00,0,PREPARADO,0,None


## 7. Transformações territoriais da camada Silver

Esta etapa prepara três estruturas territoriais:

### `silver.municipio`

Mantém os indicadores na granularidade:

ano + município + série + rede

Também acrescenta:

- nome do município;
- UF;
- região;
- sinalizações de disponibilidade da distribuição dos níveis de proficiência;
- validação do relacionamento com o diretório oficial.

### `silver.uf`

Mantém os indicadores agregados por:

ano + UF + série + rede

Também acrescenta a região e as sinalizações de disponibilidade dos níveis.

### `silver.dim_municipio`

Cria uma dimensão territorial com uma única linha por município.

Essa dimensão será utilizada para enriquecer os microdados de alunos somente pelo código oficial do município, evitando dependência de combinações específicas de ano, série e rede.

Como o modo seguro está ativo, as três transformações serão apenas registradas como preparadas.

In [9]:
sql_silver_territorio = f"""
CREATE OR REPLACE TABLE
  `{PROJECT_ID}.{SILVER_DATASET}.municipio`

PARTITION BY
  RANGE_BUCKET(ano, GENERATE_ARRAY(2023, 2025, 1))

CLUSTER BY
  sigla_uf,
  id_municipio,
  rede

AS

WITH diretorio_municipios AS (
  SELECT DISTINCT
    TRIM(id_municipio) AS id_municipio,
    TRIM(nome) AS nome_municipio,
    TRIM(sigla_uf) AS sigla_uf

  FROM
    `basedosdados.br_bd_diretorios_brasil.municipio`
)

SELECT
  CONCAT(
    CAST(m.ano AS STRING),
    '|',
    TRIM(m.id_municipio),
    '|',
    TRIM(m.serie),
    '|',
    TRIM(m.rede)
  ) AS chave_ano_municipio_serie_rede,

  m.ano,
  TRIM(m.id_municipio) AS id_municipio,
  d.nome_municipio,
  d.sigla_uf,

  CASE
    WHEN d.sigla_uf IN (
      'AC', 'AP', 'AM', 'PA', 'RO', 'RR', 'TO'
    ) THEN 'Norte'

    WHEN d.sigla_uf IN (
      'AL', 'BA', 'CE', 'MA', 'PB',
      'PE', 'PI', 'RN', 'SE'
    ) THEN 'Nordeste'

    WHEN d.sigla_uf IN (
      'DF', 'GO', 'MT', 'MS'
    ) THEN 'Centro-Oeste'

    WHEN d.sigla_uf IN (
      'ES', 'MG', 'RJ', 'SP'
    ) THEN 'Sudeste'

    WHEN d.sigla_uf IN (
      'PR', 'RS', 'SC'
    ) THEN 'Sul'

    ELSE 'Região não identificada'
  END AS regiao,

  TRIM(m.serie) AS serie,
  TRIM(m.rede) AS rede,

  m.taxa_alfabetizacao
    AS taxa_alfabetizacao_observada,

  m.media_portugues,

  m.proporcao_aluno_nivel_0,
  m.proporcao_aluno_nivel_1,
  m.proporcao_aluno_nivel_2,
  m.proporcao_aluno_nivel_3,
  m.proporcao_aluno_nivel_4,
  m.proporcao_aluno_nivel_5,
  m.proporcao_aluno_nivel_6,
  m.proporcao_aluno_nivel_7,
  m.proporcao_aluno_nivel_8,

  m.taxa_alfabetizacao IS NOT NULL
    AS taxa_observada_disponivel,

  (
    m.proporcao_aluno_nivel_0 IS NOT NULL
    AND m.proporcao_aluno_nivel_1 IS NOT NULL
    AND m.proporcao_aluno_nivel_2 IS NOT NULL
    AND m.proporcao_aluno_nivel_3 IS NOT NULL
    AND m.proporcao_aluno_nivel_4 IS NOT NULL
    AND m.proporcao_aluno_nivel_5 IS NOT NULL
    AND m.proporcao_aluno_nivel_6 IS NOT NULL
    AND m.proporcao_aluno_nivel_7 IS NOT NULL
    AND m.proporcao_aluno_nivel_8 IS NOT NULL
  ) AS distribuicao_niveis_disponivel,

  CASE
    WHEN
      m.proporcao_aluno_nivel_0 IS NOT NULL
      AND m.proporcao_aluno_nivel_1 IS NOT NULL
      AND m.proporcao_aluno_nivel_2 IS NOT NULL
      AND m.proporcao_aluno_nivel_3 IS NOT NULL
      AND m.proporcao_aluno_nivel_4 IS NOT NULL
      AND m.proporcao_aluno_nivel_5 IS NOT NULL
      AND m.proporcao_aluno_nivel_6 IS NOT NULL
      AND m.proporcao_aluno_nivel_7 IS NOT NULL
      AND m.proporcao_aluno_nivel_8 IS NOT NULL
    THEN 'disponivel'

    ELSE 'nao_disponivel_na_fonte'
  END AS status_distribuicao_niveis,

  d.id_municipio IS NOT NULL
    AS municipio_encontrado_no_diretorio,

  CURRENT_TIMESTAMP() AS processado_em

FROM
  `{PROJECT_ID}.{BRONZE_DATASET}.municipio` AS m

LEFT JOIN
  diretorio_municipios AS d
ON
  TRIM(m.id_municipio) = d.id_municipio;


CREATE OR REPLACE TABLE
  `{PROJECT_ID}.{SILVER_DATASET}.uf`

PARTITION BY
  RANGE_BUCKET(ano, GENERATE_ARRAY(2023, 2025, 1))

CLUSTER BY
  sigla_uf,
  rede

AS

SELECT
  CONCAT(
    CAST(ano AS STRING),
    '|',
    TRIM(sigla_uf),
    '|',
    TRIM(serie),
    '|',
    TRIM(rede)
  ) AS chave_ano_uf_serie_rede,

  ano,
  TRIM(sigla_uf) AS sigla_uf,

  CASE
    WHEN TRIM(sigla_uf) IN (
      'AC', 'AP', 'AM', 'PA', 'RO', 'RR', 'TO'
    ) THEN 'Norte'

    WHEN TRIM(sigla_uf) IN (
      'AL', 'BA', 'CE', 'MA', 'PB',
      'PE', 'PI', 'RN', 'SE'
    ) THEN 'Nordeste'

    WHEN TRIM(sigla_uf) IN (
      'DF', 'GO', 'MT', 'MS'
    ) THEN 'Centro-Oeste'

    WHEN TRIM(sigla_uf) IN (
      'ES', 'MG', 'RJ', 'SP'
    ) THEN 'Sudeste'

    WHEN TRIM(sigla_uf) IN (
      'PR', 'RS', 'SC'
    ) THEN 'Sul'

    ELSE 'Região não identificada'
  END AS regiao,

  TRIM(serie) AS serie,
  TRIM(rede) AS rede,

  taxa_alfabetizacao
    AS taxa_alfabetizacao_observada,

  media_portugues,

  proporcao_aluno_nivel_0,
  proporcao_aluno_nivel_1,
  proporcao_aluno_nivel_2,
  proporcao_aluno_nivel_3,
  proporcao_aluno_nivel_4,
  proporcao_aluno_nivel_5,
  proporcao_aluno_nivel_6,
  proporcao_aluno_nivel_7,
  proporcao_aluno_nivel_8,

  taxa_alfabetizacao IS NOT NULL
    AS taxa_observada_disponivel,

  (
    proporcao_aluno_nivel_0 IS NOT NULL
    AND proporcao_aluno_nivel_1 IS NOT NULL
    AND proporcao_aluno_nivel_2 IS NOT NULL
    AND proporcao_aluno_nivel_3 IS NOT NULL
    AND proporcao_aluno_nivel_4 IS NOT NULL
    AND proporcao_aluno_nivel_5 IS NOT NULL
    AND proporcao_aluno_nivel_6 IS NOT NULL
    AND proporcao_aluno_nivel_7 IS NOT NULL
    AND proporcao_aluno_nivel_8 IS NOT NULL
  ) AS distribuicao_niveis_disponivel,

  CASE
    WHEN
      proporcao_aluno_nivel_0 IS NOT NULL
      AND proporcao_aluno_nivel_1 IS NOT NULL
      AND proporcao_aluno_nivel_2 IS NOT NULL
      AND proporcao_aluno_nivel_3 IS NOT NULL
      AND proporcao_aluno_nivel_4 IS NOT NULL
      AND proporcao_aluno_nivel_5 IS NOT NULL
      AND proporcao_aluno_nivel_6 IS NOT NULL
      AND proporcao_aluno_nivel_7 IS NOT NULL
      AND proporcao_aluno_nivel_8 IS NOT NULL
    THEN 'disponivel'

    ELSE 'nao_disponivel_na_fonte'
  END AS status_distribuicao_niveis,

  CURRENT_TIMESTAMP() AS processado_em

FROM
  `{PROJECT_ID}.{BRONZE_DATASET}.uf`;


CREATE OR REPLACE TABLE
  `{PROJECT_ID}.{SILVER_DATASET}.dim_municipio`

CLUSTER BY
  sigla_uf

AS

WITH municipios AS (
  SELECT
    TRIM(id_municipio) AS id_municipio,
    TRIM(nome) AS nome_municipio,
    TRIM(sigla_uf) AS sigla_uf,

    CASE
      WHEN TRIM(sigla_uf) IN (
        'AC', 'AP', 'AM', 'PA', 'RO', 'RR', 'TO'
      ) THEN 'Norte'

      WHEN TRIM(sigla_uf) IN (
        'AL', 'BA', 'CE', 'MA', 'PB',
        'PE', 'PI', 'RN', 'SE'
      ) THEN 'Nordeste'

      WHEN TRIM(sigla_uf) IN (
        'DF', 'GO', 'MT', 'MS'
      ) THEN 'Centro-Oeste'

      WHEN TRIM(sigla_uf) IN (
        'ES', 'MG', 'RJ', 'SP'
      ) THEN 'Sudeste'

      WHEN TRIM(sigla_uf) IN (
        'PR', 'RS', 'SC'
      ) THEN 'Sul'

      ELSE 'Região não identificada'
    END AS regiao

  FROM
    `basedosdados.br_bd_diretorios_brasil.municipio`

  WHERE
    id_municipio IS NOT NULL
)

SELECT
  id_municipio,
  nome_municipio,
  sigla_uf,
  regiao,
  CURRENT_TIMESTAMP() AS processado_em

FROM
  municipios

QUALIFY
  ROW_NUMBER() OVER (
    PARTITION BY id_municipio
    ORDER BY nome_municipio, sigla_uf
  ) = 1;
"""

executar_sql(
    camada="SILVER",
    etapa="criar_silver_territorio",
    sql=sql_silver_territorio,
    executar=EXECUTE_PIPELINE,
)

display(
    pd.DataFrame(pipeline_logs).tail(1)
)

[PREPARADO] SILVER | criar_silver_territorio


,execution_id,camada,etapa,inicio_utc,fim_utc,duracao_segundos,status,bytes_processados,mensagem_erro
9,edd46773-51b2-4145-b489-09a398e33c9a,SILVER,criar_silver_territorio,2026-07-12T15:34:13.126261+00:00,2026-07-12T15:34:13.126287+00:00,0,PREPARADO,0,None


## 8. Transformações das metas educacionais na camada Silver

Esta etapa prepara três tabelas:

### `silver.meta_alfabetizacao_brasil`

Mantém os resultados observados, a participação e as metas nacionais de 2024 a 2030.

### `silver.meta_alfabetizacao_uf`

Mantém os resultados e metas por unidade federativa, acrescentando a região correspondente.

### `silver.meta_alfabetizacao_municipio`

Mantém os resultados e metas municipais, enriquecendo os registros com:

- nome do município;
- UF;
- região;
- validação do relacionamento territorial.

Os valores ausentes existentes na fonte serão preservados.

Para cada meta, serão criados indicadores de disponibilidade. Também será sinalizado se a série de metas de 2024 a 2030 está completa.

Como o modo seguro permanece ativo, as transformações serão apenas registradas como preparadas.

In [10]:
sql_silver_metas = f"""
CREATE OR REPLACE TABLE
  `{PROJECT_ID}.{SILVER_DATASET}.meta_alfabetizacao_brasil`

AS

SELECT
  CONCAT(
    CAST(ano AS STRING),
    '|',
    TRIM(rede)
  ) AS chave_ano_rede,

  ano,
  TRIM(rede) AS rede,

  taxa_alfabetizacao
    AS taxa_alfabetizacao_observada,

  percentual_participacao,

  meta_alfabetizacao_2024,
  meta_alfabetizacao_2025,
  meta_alfabetizacao_2026,
  meta_alfabetizacao_2027,
  meta_alfabetizacao_2028,
  meta_alfabetizacao_2029,
  meta_alfabetizacao_2030,

  taxa_alfabetizacao IS NOT NULL
    AS taxa_observada_disponivel,

  percentual_participacao IS NOT NULL
    AS participacao_disponivel,

  meta_alfabetizacao_2024 IS NOT NULL
    AS meta_2024_disponivel,

  meta_alfabetizacao_2025 IS NOT NULL
    AS meta_2025_disponivel,

  meta_alfabetizacao_2026 IS NOT NULL
    AS meta_2026_disponivel,

  meta_alfabetizacao_2027 IS NOT NULL
    AS meta_2027_disponivel,

  meta_alfabetizacao_2028 IS NOT NULL
    AS meta_2028_disponivel,

  meta_alfabetizacao_2029 IS NOT NULL
    AS meta_2029_disponivel,

  meta_alfabetizacao_2030 IS NOT NULL
    AS meta_2030_disponivel,

  (
    meta_alfabetizacao_2024 IS NOT NULL
    AND meta_alfabetizacao_2025 IS NOT NULL
    AND meta_alfabetizacao_2026 IS NOT NULL
    AND meta_alfabetizacao_2027 IS NOT NULL
    AND meta_alfabetizacao_2028 IS NOT NULL
    AND meta_alfabetizacao_2029 IS NOT NULL
    AND meta_alfabetizacao_2030 IS NOT NULL
  ) AS serie_metas_completa,

  CASE
    WHEN
      meta_alfabetizacao_2024 IS NOT NULL
      AND meta_alfabetizacao_2025 IS NOT NULL
      AND meta_alfabetizacao_2026 IS NOT NULL
      AND meta_alfabetizacao_2027 IS NOT NULL
      AND meta_alfabetizacao_2028 IS NOT NULL
      AND meta_alfabetizacao_2029 IS NOT NULL
      AND meta_alfabetizacao_2030 IS NOT NULL
    THEN 'completa'

    ELSE 'incompleta_na_fonte'
  END AS status_serie_metas,

  CURRENT_TIMESTAMP() AS processado_em

FROM
  `{PROJECT_ID}.{BRONZE_DATASET}.meta_alfabetizacao_brasil`;


CREATE OR REPLACE TABLE
  `{PROJECT_ID}.{SILVER_DATASET}.meta_alfabetizacao_uf`

AS

SELECT
  CONCAT(
    CAST(ano AS STRING),
    '|',
    TRIM(sigla_uf),
    '|',
    TRIM(rede)
  ) AS chave_ano_uf_rede,

  ano,
  TRIM(sigla_uf) AS sigla_uf,

  CASE
    WHEN TRIM(sigla_uf) IN (
      'AC', 'AP', 'AM', 'PA', 'RO', 'RR', 'TO'
    ) THEN 'Norte'

    WHEN TRIM(sigla_uf) IN (
      'AL', 'BA', 'CE', 'MA', 'PB',
      'PE', 'PI', 'RN', 'SE'
    ) THEN 'Nordeste'

    WHEN TRIM(sigla_uf) IN (
      'DF', 'GO', 'MT', 'MS'
    ) THEN 'Centro-Oeste'

    WHEN TRIM(sigla_uf) IN (
      'ES', 'MG', 'RJ', 'SP'
    ) THEN 'Sudeste'

    WHEN TRIM(sigla_uf) IN (
      'PR', 'RS', 'SC'
    ) THEN 'Sul'

    ELSE 'Região não identificada'
  END AS regiao,

  TRIM(rede) AS rede,

  taxa_alfabetizacao
    AS taxa_alfabetizacao_observada,

  percentual_participacao,

  meta_alfabetizacao_2024,
  meta_alfabetizacao_2025,
  meta_alfabetizacao_2026,
  meta_alfabetizacao_2027,
  meta_alfabetizacao_2028,
  meta_alfabetizacao_2029,
  meta_alfabetizacao_2030,

  taxa_alfabetizacao IS NOT NULL
    AS taxa_observada_disponivel,

  percentual_participacao IS NOT NULL
    AS participacao_disponivel,

  meta_alfabetizacao_2024 IS NOT NULL
    AS meta_2024_disponivel,

  meta_alfabetizacao_2025 IS NOT NULL
    AS meta_2025_disponivel,

  meta_alfabetizacao_2026 IS NOT NULL
    AS meta_2026_disponivel,

  meta_alfabetizacao_2027 IS NOT NULL
    AS meta_2027_disponivel,

  meta_alfabetizacao_2028 IS NOT NULL
    AS meta_2028_disponivel,

  meta_alfabetizacao_2029 IS NOT NULL
    AS meta_2029_disponivel,

  meta_alfabetizacao_2030 IS NOT NULL
    AS meta_2030_disponivel,

  (
    meta_alfabetizacao_2024 IS NOT NULL
    AND meta_alfabetizacao_2025 IS NOT NULL
    AND meta_alfabetizacao_2026 IS NOT NULL
    AND meta_alfabetizacao_2027 IS NOT NULL
    AND meta_alfabetizacao_2028 IS NOT NULL
    AND meta_alfabetizacao_2029 IS NOT NULL
    AND meta_alfabetizacao_2030 IS NOT NULL
  ) AS serie_metas_completa,

  CASE
    WHEN
      meta_alfabetizacao_2024 IS NOT NULL
      AND meta_alfabetizacao_2025 IS NOT NULL
      AND meta_alfabetizacao_2026 IS NOT NULL
      AND meta_alfabetizacao_2027 IS NOT NULL
      AND meta_alfabetizacao_2028 IS NOT NULL
      AND meta_alfabetizacao_2029 IS NOT NULL
      AND meta_alfabetizacao_2030 IS NOT NULL
    THEN 'completa'

    ELSE 'incompleta_na_fonte'
  END AS status_serie_metas,

  CURRENT_TIMESTAMP() AS processado_em

FROM
  `{PROJECT_ID}.{BRONZE_DATASET}.meta_alfabetizacao_uf`;


CREATE OR REPLACE TABLE
  `{PROJECT_ID}.{SILVER_DATASET}.meta_alfabetizacao_municipio`

AS

WITH diretorio_municipios AS (
  SELECT DISTINCT
    TRIM(id_municipio) AS id_municipio,
    TRIM(nome) AS nome_municipio,
    TRIM(sigla_uf) AS sigla_uf

  FROM
    `basedosdados.br_bd_diretorios_brasil.municipio`
)

SELECT
  CONCAT(
    CAST(m.ano AS STRING),
    '|',
    TRIM(m.id_municipio),
    '|',
    TRIM(m.rede)
  ) AS chave_ano_municipio_rede,

  m.ano,
  TRIM(m.id_municipio) AS id_municipio,
  d.nome_municipio,
  d.sigla_uf,

  CASE
    WHEN d.sigla_uf IN (
      'AC', 'AP', 'AM', 'PA', 'RO', 'RR', 'TO'
    ) THEN 'Norte'

    WHEN d.sigla_uf IN (
      'AL', 'BA', 'CE', 'MA', 'PB',
      'PE', 'PI', 'RN', 'SE'
    ) THEN 'Nordeste'

    WHEN d.sigla_uf IN (
      'DF', 'GO', 'MT', 'MS'
    ) THEN 'Centro-Oeste'

    WHEN d.sigla_uf IN (
      'ES', 'MG', 'RJ', 'SP'
    ) THEN 'Sudeste'

    WHEN d.sigla_uf IN (
      'PR', 'RS', 'SC'
    ) THEN 'Sul'

    ELSE 'Região não identificada'
  END AS regiao,

  TRIM(m.rede) AS rede,

  m.taxa_alfabetizacao
    AS taxa_alfabetizacao_observada,

  m.nivel_alfabetizacao,
  m.percentual_participacao,

  m.meta_alfabetizacao_2024,
  m.meta_alfabetizacao_2025,
  m.meta_alfabetizacao_2026,
  m.meta_alfabetizacao_2027,
  m.meta_alfabetizacao_2028,
  m.meta_alfabetizacao_2029,
  m.meta_alfabetizacao_2030,

  m.taxa_alfabetizacao IS NOT NULL
    AS taxa_observada_disponivel,

  m.nivel_alfabetizacao IS NOT NULL
    AS nivel_alfabetizacao_disponivel,

  m.percentual_participacao IS NOT NULL
    AS participacao_disponivel,

  m.meta_alfabetizacao_2024 IS NOT NULL
    AS meta_2024_disponivel,

  m.meta_alfabetizacao_2025 IS NOT NULL
    AS meta_2025_disponivel,

  m.meta_alfabetizacao_2026 IS NOT NULL
    AS meta_2026_disponivel,

  m.meta_alfabetizacao_2027 IS NOT NULL
    AS meta_2027_disponivel,

  m.meta_alfabetizacao_2028 IS NOT NULL
    AS meta_2028_disponivel,

  m.meta_alfabetizacao_2029 IS NOT NULL
    AS meta_2029_disponivel,

  m.meta_alfabetizacao_2030 IS NOT NULL
    AS meta_2030_disponivel,

  (
    m.meta_alfabetizacao_2024 IS NOT NULL
    AND m.meta_alfabetizacao_2025 IS NOT NULL
    AND m.meta_alfabetizacao_2026 IS NOT NULL
    AND m.meta_alfabetizacao_2027 IS NOT NULL
    AND m.meta_alfabetizacao_2028 IS NOT NULL
    AND m.meta_alfabetizacao_2029 IS NOT NULL
    AND m.meta_alfabetizacao_2030 IS NOT NULL
  ) AS serie_metas_completa,

  CASE
    WHEN
      m.meta_alfabetizacao_2024 IS NOT NULL
      AND m.meta_alfabetizacao_2025 IS NOT NULL
      AND m.meta_alfabetizacao_2026 IS NOT NULL
      AND m.meta_alfabetizacao_2027 IS NOT NULL
      AND m.meta_alfabetizacao_2028 IS NOT NULL
      AND m.meta_alfabetizacao_2029 IS NOT NULL
      AND m.meta_alfabetizacao_2030 IS NOT NULL
    THEN 'completa'

    ELSE 'incompleta_na_fonte'
  END AS status_serie_metas,

  d.id_municipio IS NOT NULL
    AS municipio_encontrado_no_diretorio,

  CURRENT_TIMESTAMP() AS processado_em

FROM
  `{PROJECT_ID}.{BRONZE_DATASET}.meta_alfabetizacao_municipio` AS m

LEFT JOIN
  diretorio_municipios AS d
ON
  TRIM(m.id_municipio) = d.id_municipio;
"""

executar_sql(
    camada="SILVER",
    etapa="criar_silver_metas",
    sql=sql_silver_metas,
    executar=EXECUTE_PIPELINE,
)

display(
    pd.DataFrame(pipeline_logs).tail(1)
)

[PREPARADO] SILVER | criar_silver_metas


,execution_id,camada,etapa,inicio_utc,fim_utc,duracao_segundos,status,bytes_processados,mensagem_erro
10,edd46773-51b2-4145-b489-09a398e33c9a,SILVER,criar_silver_metas,2026-07-12T15:36:04.963627+00:00,2026-07-12T15:36:04.963654+00:00,0,PREPARADO,0,None


## 9. Quality gate consolidado da camada Silver

Esta etapa valida todas as estruturas tratadas da camada Silver antes de permitir a construção da Gold.

Serão verificadas:

- correspondência de volume entre Bronze e Silver;
- unicidade das chaves;
- consistência das avaliações dos alunos;
- coerência da classificação com o corte de 743 pontos;
- integridade dos relacionamentos territoriais;
- disponibilidade dos níveis de proficiência;
- preservação dos valores ausentes existentes na fonte;
- integridade das metas nacionais, estaduais e municipais;
- unicidade e completude da dimensão municipal.

O quality gate é executado somente em modo de leitura.

Caso alguma tabela apresente divergência, o pipeline será interrompido antes da camada Gold.

In [11]:
sql_validar_silver = f"""
WITH

alunos_bronze AS (
  SELECT
    COUNT(*) AS total,
    COUNTIF(proficiencia IS NULL) AS nulos_proficiencia,
    COUNTIF(peso_aluno IS NULL) AS nulos_peso
  FROM
    `{PROJECT_ID}.{BRONZE_DATASET}.alunos`
),

alunos_silver AS (
  SELECT
    COUNT(*) AS total,
    COUNT(DISTINCT chave_ano_aluno) AS chaves_distintas,
    COUNTIF(status_avaliacao = 'situacao_inconsistente')
      + COUNTIF(classificacao_coerente_743 IS FALSE)
      AS problemas_integridade,
    COUNTIF(proficiencia IS NULL) AS nulos_proficiencia,
    COUNTIF(peso_aluno IS NULL) AS nulos_peso
  FROM
    `{PROJECT_ID}.{SILVER_DATASET}.alunos`
),

municipio_bronze AS (
  SELECT
    COUNT(*) AS total,
    COUNTIF(
      proporcao_aluno_nivel_0 IS NULL
      OR proporcao_aluno_nivel_1 IS NULL
      OR proporcao_aluno_nivel_2 IS NULL
      OR proporcao_aluno_nivel_3 IS NULL
      OR proporcao_aluno_nivel_4 IS NULL
      OR proporcao_aluno_nivel_5 IS NULL
      OR proporcao_aluno_nivel_6 IS NULL
      OR proporcao_aluno_nivel_7 IS NULL
      OR proporcao_aluno_nivel_8 IS NULL
    ) AS distribuicoes_indisponiveis
  FROM
    `{PROJECT_ID}.{BRONZE_DATASET}.municipio`
),

municipio_silver AS (
  SELECT
    COUNT(*) AS total,
    COUNT(DISTINCT chave_ano_municipio_serie_rede)
      AS chaves_distintas,
    COUNTIF(NOT municipio_encontrado_no_diretorio)
      + COUNTIF(regiao = 'Região não identificada')
      + COUNTIF(
          ano = 2023
          AND distribuicao_niveis_disponivel
        )
      + COUNTIF(
          ano = 2024
          AND NOT distribuicao_niveis_disponivel
        )
      AS problemas_integridade,
    COUNTIF(NOT distribuicao_niveis_disponivel)
      AS distribuicoes_indisponiveis
  FROM
    `{PROJECT_ID}.{SILVER_DATASET}.municipio`
),

uf_bronze AS (
  SELECT
    COUNT(*) AS total,
    COUNTIF(
      proporcao_aluno_nivel_0 IS NULL
      OR proporcao_aluno_nivel_1 IS NULL
      OR proporcao_aluno_nivel_2 IS NULL
      OR proporcao_aluno_nivel_3 IS NULL
      OR proporcao_aluno_nivel_4 IS NULL
      OR proporcao_aluno_nivel_5 IS NULL
      OR proporcao_aluno_nivel_6 IS NULL
      OR proporcao_aluno_nivel_7 IS NULL
      OR proporcao_aluno_nivel_8 IS NULL
    ) AS distribuicoes_indisponiveis
  FROM
    `{PROJECT_ID}.{BRONZE_DATASET}.uf`
),

uf_silver AS (
  SELECT
    COUNT(*) AS total,
    COUNT(DISTINCT chave_ano_uf_serie_rede)
      AS chaves_distintas,
    COUNTIF(regiao = 'Região não identificada')
      + COUNTIF(
          ano = 2023
          AND distribuicao_niveis_disponivel
        )
      + COUNTIF(
          ano = 2024
          AND NOT distribuicao_niveis_disponivel
        )
      AS problemas_integridade,
    COUNTIF(NOT distribuicao_niveis_disponivel)
      AS distribuicoes_indisponiveis
  FROM
    `{PROJECT_ID}.{SILVER_DATASET}.uf`
),

meta_brasil_bronze AS (
  SELECT
    COUNT(*) AS total,
    COUNTIF(taxa_alfabetizacao IS NULL) AS n_taxa,
    COUNTIF(percentual_participacao IS NULL) AS n_participacao,
    COUNTIF(meta_alfabetizacao_2024 IS NULL) AS n_2024,
    COUNTIF(meta_alfabetizacao_2025 IS NULL) AS n_2025,
    COUNTIF(meta_alfabetizacao_2026 IS NULL) AS n_2026,
    COUNTIF(meta_alfabetizacao_2027 IS NULL) AS n_2027,
    COUNTIF(meta_alfabetizacao_2028 IS NULL) AS n_2028,
    COUNTIF(meta_alfabetizacao_2029 IS NULL) AS n_2029,
    COUNTIF(meta_alfabetizacao_2030 IS NULL) AS n_2030
  FROM
    `{PROJECT_ID}.{BRONZE_DATASET}.meta_alfabetizacao_brasil`
),

meta_brasil_silver AS (
  SELECT
    COUNT(*) AS total,
    COUNT(DISTINCT chave_ano_rede) AS chaves_distintas,
    COUNTIF(chave_ano_rede IS NULL OR chave_ano_rede = '')
      AS problemas_integridade,
    COUNTIF(taxa_alfabetizacao_observada IS NULL) AS n_taxa,
    COUNTIF(percentual_participacao IS NULL) AS n_participacao,
    COUNTIF(meta_alfabetizacao_2024 IS NULL) AS n_2024,
    COUNTIF(meta_alfabetizacao_2025 IS NULL) AS n_2025,
    COUNTIF(meta_alfabetizacao_2026 IS NULL) AS n_2026,
    COUNTIF(meta_alfabetizacao_2027 IS NULL) AS n_2027,
    COUNTIF(meta_alfabetizacao_2028 IS NULL) AS n_2028,
    COUNTIF(meta_alfabetizacao_2029 IS NULL) AS n_2029,
    COUNTIF(meta_alfabetizacao_2030 IS NULL) AS n_2030
  FROM
    `{PROJECT_ID}.{SILVER_DATASET}.meta_alfabetizacao_brasil`
),

meta_uf_bronze AS (
  SELECT
    COUNT(*) AS total,
    COUNTIF(taxa_alfabetizacao IS NULL) AS n_taxa,
    COUNTIF(percentual_participacao IS NULL) AS n_participacao,
    COUNTIF(meta_alfabetizacao_2024 IS NULL) AS n_2024,
    COUNTIF(meta_alfabetizacao_2025 IS NULL) AS n_2025,
    COUNTIF(meta_alfabetizacao_2026 IS NULL) AS n_2026,
    COUNTIF(meta_alfabetizacao_2027 IS NULL) AS n_2027,
    COUNTIF(meta_alfabetizacao_2028 IS NULL) AS n_2028,
    COUNTIF(meta_alfabetizacao_2029 IS NULL) AS n_2029,
    COUNTIF(meta_alfabetizacao_2030 IS NULL) AS n_2030
  FROM
    `{PROJECT_ID}.{BRONZE_DATASET}.meta_alfabetizacao_uf`
),

meta_uf_silver AS (
  SELECT
    COUNT(*) AS total,
    COUNT(DISTINCT chave_ano_uf_rede) AS chaves_distintas,
    COUNTIF(regiao = 'Região não identificada')
      AS problemas_integridade,
    COUNTIF(taxa_alfabetizacao_observada IS NULL) AS n_taxa,
    COUNTIF(percentual_participacao IS NULL) AS n_participacao,
    COUNTIF(meta_alfabetizacao_2024 IS NULL) AS n_2024,
    COUNTIF(meta_alfabetizacao_2025 IS NULL) AS n_2025,
    COUNTIF(meta_alfabetizacao_2026 IS NULL) AS n_2026,
    COUNTIF(meta_alfabetizacao_2027 IS NULL) AS n_2027,
    COUNTIF(meta_alfabetizacao_2028 IS NULL) AS n_2028,
    COUNTIF(meta_alfabetizacao_2029 IS NULL) AS n_2029,
    COUNTIF(meta_alfabetizacao_2030 IS NULL) AS n_2030
  FROM
    `{PROJECT_ID}.{SILVER_DATASET}.meta_alfabetizacao_uf`
),

meta_municipio_bronze AS (
  SELECT
    COUNT(*) AS total,
    COUNTIF(taxa_alfabetizacao IS NULL) AS n_taxa,
    COUNTIF(percentual_participacao IS NULL) AS n_participacao,
    COUNTIF(meta_alfabetizacao_2024 IS NULL) AS n_2024,
    COUNTIF(meta_alfabetizacao_2025 IS NULL) AS n_2025,
    COUNTIF(meta_alfabetizacao_2026 IS NULL) AS n_2026,
    COUNTIF(meta_alfabetizacao_2027 IS NULL) AS n_2027,
    COUNTIF(meta_alfabetizacao_2028 IS NULL) AS n_2028,
    COUNTIF(meta_alfabetizacao_2029 IS NULL) AS n_2029,
    COUNTIF(meta_alfabetizacao_2030 IS NULL) AS n_2030
  FROM
    `{PROJECT_ID}.{BRONZE_DATASET}.meta_alfabetizacao_municipio`
),

meta_municipio_silver AS (
  SELECT
    COUNT(*) AS total,
    COUNT(DISTINCT chave_ano_municipio_rede)
      AS chaves_distintas,
    COUNTIF(NOT municipio_encontrado_no_diretorio)
      + COUNTIF(regiao = 'Região não identificada')
      AS problemas_integridade,
    COUNTIF(taxa_alfabetizacao_observada IS NULL) AS n_taxa,
    COUNTIF(percentual_participacao IS NULL) AS n_participacao,
    COUNTIF(meta_alfabetizacao_2024 IS NULL) AS n_2024,
    COUNTIF(meta_alfabetizacao_2025 IS NULL) AS n_2025,
    COUNTIF(meta_alfabetizacao_2026 IS NULL) AS n_2026,
    COUNTIF(meta_alfabetizacao_2027 IS NULL) AS n_2027,
    COUNTIF(meta_alfabetizacao_2028 IS NULL) AS n_2028,
    COUNTIF(meta_alfabetizacao_2029 IS NULL) AS n_2029,
    COUNTIF(meta_alfabetizacao_2030 IS NULL) AS n_2030
  FROM
    `{PROJECT_ID}.{SILVER_DATASET}.meta_alfabetizacao_municipio`
),

dim_fonte AS (
  SELECT
    COUNT(DISTINCT TRIM(id_municipio)) AS total
  FROM
    `basedosdados.br_bd_diretorios_brasil.municipio`
  WHERE
    id_municipio IS NOT NULL
),

dim_silver AS (
  SELECT
    COUNT(*) AS total,
    COUNT(DISTINCT id_municipio) AS chaves_distintas,
    COUNTIF(
      id_municipio IS NULL
      OR nome_municipio IS NULL
      OR sigla_uf IS NULL
      OR regiao IS NULL
      OR regiao = 'Região não identificada'
    ) AS problemas_integridade
  FROM
    `{PROJECT_ID}.{SILVER_DATASET}.dim_municipio`
)

SELECT
  'alunos' AS tabela,
  b.total AS linhas_fonte,
  s.total AS linhas_silver,
  b.total = s.total AS contagem_confere,
  s.chaves_distintas,
  s.total = s.chaves_distintas AS chave_unica_confere,
  s.problemas_integridade,
  (
    b.nulos_proficiencia = s.nulos_proficiencia
    AND b.nulos_peso = s.nulos_peso
  ) AS ausencias_preservadas
FROM alunos_bronze b
CROSS JOIN alunos_silver s

UNION ALL

SELECT
  'municipio',
  b.total,
  s.total,
  b.total = s.total,
  s.chaves_distintas,
  s.total = s.chaves_distintas,
  s.problemas_integridade,
  b.distribuicoes_indisponiveis
    = s.distribuicoes_indisponiveis
FROM municipio_bronze b
CROSS JOIN municipio_silver s

UNION ALL

SELECT
  'uf',
  b.total,
  s.total,
  b.total = s.total,
  s.chaves_distintas,
  s.total = s.chaves_distintas,
  s.problemas_integridade,
  b.distribuicoes_indisponiveis
    = s.distribuicoes_indisponiveis
FROM uf_bronze b
CROSS JOIN uf_silver s

UNION ALL

SELECT
  'meta_alfabetizacao_brasil',
  b.total,
  s.total,
  b.total = s.total,
  s.chaves_distintas,
  s.total = s.chaves_distintas,
  s.problemas_integridade,
  (
    b.n_taxa = s.n_taxa
    AND b.n_participacao = s.n_participacao
    AND b.n_2024 = s.n_2024
    AND b.n_2025 = s.n_2025
    AND b.n_2026 = s.n_2026
    AND b.n_2027 = s.n_2027
    AND b.n_2028 = s.n_2028
    AND b.n_2029 = s.n_2029
    AND b.n_2030 = s.n_2030
  )
FROM meta_brasil_bronze b
CROSS JOIN meta_brasil_silver s

UNION ALL

SELECT
  'meta_alfabetizacao_uf',
  b.total,
  s.total,
  b.total = s.total,
  s.chaves_distintas,
  s.total = s.chaves_distintas,
  s.problemas_integridade,
  (
    b.n_taxa = s.n_taxa
    AND b.n_participacao = s.n_participacao
    AND b.n_2024 = s.n_2024
    AND b.n_2025 = s.n_2025
    AND b.n_2026 = s.n_2026
    AND b.n_2027 = s.n_2027
    AND b.n_2028 = s.n_2028
    AND b.n_2029 = s.n_2029
    AND b.n_2030 = s.n_2030
  )
FROM meta_uf_bronze b
CROSS JOIN meta_uf_silver s

UNION ALL

SELECT
  'meta_alfabetizacao_municipio',
  b.total,
  s.total,
  b.total = s.total,
  s.chaves_distintas,
  s.total = s.chaves_distintas,
  s.problemas_integridade,
  (
    b.n_taxa = s.n_taxa
    AND b.n_participacao = s.n_participacao
    AND b.n_2024 = s.n_2024
    AND b.n_2025 = s.n_2025
    AND b.n_2026 = s.n_2026
    AND b.n_2027 = s.n_2027
    AND b.n_2028 = s.n_2028
    AND b.n_2029 = s.n_2029
    AND b.n_2030 = s.n_2030
  )
FROM meta_municipio_bronze b
CROSS JOIN meta_municipio_silver s

UNION ALL

SELECT
  'dim_municipio',
  f.total,
  s.total,
  f.total = s.total,
  s.chaves_distintas,
  s.total = s.chaves_distintas,
  s.problemas_integridade,
  s.problemas_integridade = 0
FROM dim_fonte f
CROSS JOIN dim_silver s

ORDER BY
  tabela
"""

silver_validation_df = (
    client
    .query(
        sql_validar_silver,
        location=LOCATION,
    )
    .result()
    .to_dataframe()
)

display(silver_validation_df)

silver_quality_gate = bool(
    silver_validation_df["contagem_confere"].all()
    and silver_validation_df["chave_unica_confere"].all()
    and (
        silver_validation_df["problemas_integridade"] == 0
    ).all()
    and silver_validation_df["ausencias_preservadas"].all()
)

print(
    "\nStatus do quality gate Silver:",
    "APROVADO" if silver_quality_gate else "REPROVADO",
)

if not silver_quality_gate:
    raise RuntimeError(
        "Quality gate da Silver reprovado. "
        "A Gold não deve ser processada."
    )

,tabela,linhas_fonte,linhas_silver,contagem_confere,chaves_distintas,chave_unica_confere,problemas_integridade,ausencias_preservadas
0,alunos,3867999,3867999,True,3867999,True,0,True
1,dim_municipio,5571,5571,True,5571,True,0,True
2,meta_alfabetizacao_brasil,3,3,True,3,True,0,True
3,meta_alfabetizacao_municipio,10704,10704,True,10704,True,0,True
4,meta_alfabetizacao_uf,81,81,True,81,True,0,True
5,municipio,23995,23995,True,23995,True,0,True
6,uf,145,145,True,145,True,0,True



Status do quality gate Silver: APROVADO


## 10. Indicador municipal da camada Gold

Esta etapa prepara a tabela `gold.indicador_municipio`.

A tabela consolida, para cada ano, município e rede:

- resultado observado de alfabetização;
- meta correspondente ao mesmo ano;
- diferença entre resultado e meta;
- distância necessária para atingir a meta;
- excedente acima da meta;
- situação de cumprimento;
- participação e nível de alfabetização;
- informações territoriais.

A meta será selecionada dinamicamente de acordo com o ano do registro.

Exemplo:

- para um registro de 2024, será utilizada a meta de 2024;
- para um registro de 2025, será utilizada a meta de 2025;
- quando não existir meta para o ano, o valor será preservado como ausente.

As diferenças serão arredondadas para duas casas decimais, evitando divergências causadas por precisão de números de ponto flutuante.

A tabela será particionada por ano e clusterizada por UF, município e rede.

Como o modo seguro permanece ativo, a transformação será apenas registrada como preparada.

In [12]:
sql_gold_indicador_municipio = f"""
CREATE OR REPLACE TABLE
  `{PROJECT_ID}.{GOLD_DATASET}.indicador_municipio`

PARTITION BY
  RANGE_BUCKET(ano, GENERATE_ARRAY(2023, 2031, 1))

CLUSTER BY
  sigla_uf,
  id_municipio,
  rede

AS

WITH metas_alinhadas AS (
  SELECT
    chave_ano_municipio_rede,
    ano,
    id_municipio,
    nome_municipio,
    sigla_uf,
    regiao,
    rede,

    ROUND(
      CAST(taxa_alfabetizacao_observada AS NUMERIC),
      2
    ) AS resultado_alfabetizacao,

    nivel_alfabetizacao,

    ROUND(
      CAST(percentual_participacao AS NUMERIC),
      2
    ) AS percentual_participacao,

    CASE ano
      WHEN 2024 THEN meta_alfabetizacao_2024
      WHEN 2025 THEN meta_alfabetizacao_2025
      WHEN 2026 THEN meta_alfabetizacao_2026
      WHEN 2027 THEN meta_alfabetizacao_2027
      WHEN 2028 THEN meta_alfabetizacao_2028
      WHEN 2029 THEN meta_alfabetizacao_2029
      WHEN 2030 THEN meta_alfabetizacao_2030
      ELSE NULL
    END AS meta_alfabetizacao_ano,

    taxa_observada_disponivel,
    participacao_disponivel,
    serie_metas_completa,
    status_serie_metas,
    municipio_encontrado_no_diretorio

  FROM
    `{PROJECT_ID}.{SILVER_DATASET}.meta_alfabetizacao_municipio`
),

metricas AS (
  SELECT
    chave_ano_municipio_rede,
    ano,
    id_municipio,
    nome_municipio,
    sigla_uf,
    regiao,
    rede,
    resultado_alfabetizacao,
    nivel_alfabetizacao,
    percentual_participacao,

    ROUND(
      CAST(meta_alfabetizacao_ano AS NUMERIC),
      2
    ) AS meta_alfabetizacao_ano,

    taxa_observada_disponivel,
    participacao_disponivel,
    serie_metas_completa,
    status_serie_metas,
    municipio_encontrado_no_diretorio

  FROM
    metas_alinhadas
)

SELECT
  chave_ano_municipio_rede,
  ano,
  id_municipio,
  nome_municipio,
  sigla_uf,
  regiao,
  rede,
  resultado_alfabetizacao,
  meta_alfabetizacao_ano,

  CASE
    WHEN
      resultado_alfabetizacao IS NOT NULL
      AND meta_alfabetizacao_ano IS NOT NULL
    THEN ROUND(
      resultado_alfabetizacao
      - meta_alfabetizacao_ano,
      2
    )

    ELSE NULL
  END AS diferenca_resultado_meta,

  CASE
    WHEN
      resultado_alfabetizacao IS NOT NULL
      AND meta_alfabetizacao_ano IS NOT NULL
    THEN ROUND(
      GREATEST(
        meta_alfabetizacao_ano
        - resultado_alfabetizacao,
        0
      ),
      2
    )

    ELSE NULL
  END AS gap_para_meta,

  CASE
    WHEN
      resultado_alfabetizacao IS NOT NULL
      AND meta_alfabetizacao_ano IS NOT NULL
    THEN ROUND(
      GREATEST(
        resultado_alfabetizacao
        - meta_alfabetizacao_ano,
        0
      ),
      2
    )

    ELSE NULL
  END AS excedente_acima_meta,

  CASE
    WHEN resultado_alfabetizacao IS NULL
      THEN 'resultado_indisponivel'

    WHEN meta_alfabetizacao_ano IS NULL
      THEN 'meta_indisponivel_para_o_ano'

    WHEN resultado_alfabetizacao >= meta_alfabetizacao_ano
      THEN 'meta_atingida'

    ELSE 'meta_nao_atingida'
  END AS status_meta,

  nivel_alfabetizacao,
  percentual_participacao,
  taxa_observada_disponivel,
  participacao_disponivel,
  serie_metas_completa,
  status_serie_metas,
  municipio_encontrado_no_diretorio,
  CURRENT_TIMESTAMP() AS processado_em

FROM
  metricas
"""

executar_sql(
    camada="GOLD",
    etapa="criar_gold_indicador_municipio",
    sql=sql_gold_indicador_municipio,
    executar=EXECUTE_PIPELINE,
)

display(
    pd.DataFrame(pipeline_logs).tail(1)
)

[PREPARADO] GOLD | criar_gold_indicador_municipio


,execution_id,camada,etapa,inicio_utc,fim_utc,duracao_segundos,status,bytes_processados,mensagem_erro
11,edd46773-51b2-4145-b489-09a398e33c9a,GOLD,criar_gold_indicador_municipio,2026-07-12T15:40:49.747779+00:00,2026-07-12T15:40:49.747796+00:00,0,PREPARADO,0,None


## 11. Indicadores estaduais e nacionais da camada Gold

Esta etapa prepara duas tabelas analíticas:

### `gold.indicador_uf`

Consolida, para cada ano, unidade federativa e rede:

- resultado observado de alfabetização;
- meta correspondente ao mesmo ano;
- diferença entre resultado e meta;
- distância necessária para atingir a meta;
- excedente acima da meta;
- situação de cumprimento;
- percentual de participação;
- região geográfica.

### `gold.indicador_brasil`

Aplica as mesmas regras na granularidade nacional por ano e rede.

A meta será selecionada dinamicamente conforme o ano do registro. Quando o resultado ou a meta estiver ausente na fonte, essa ausência será preservada.

Como essas tabelas possuem poucos registros, elas não serão particionadas. Essa decisão evita particionamento desnecessário e contribui para a simplicidade e o controle de custos.

Como o modo seguro permanece ativo, as transformações serão apenas registradas como preparadas.

In [13]:
sql_gold_indicadores_uf_brasil = f"""
CREATE OR REPLACE TABLE
  `{PROJECT_ID}.{GOLD_DATASET}.indicador_uf`

CLUSTER BY
  sigla_uf,
  rede

AS

WITH metas_alinhadas AS (
  SELECT
    chave_ano_uf_rede,
    ano,
    sigla_uf,
    regiao,
    rede,

    ROUND(
      CAST(taxa_alfabetizacao_observada AS NUMERIC),
      2
    ) AS resultado_alfabetizacao,

    ROUND(
      CAST(percentual_participacao AS NUMERIC),
      2
    ) AS percentual_participacao,

    CASE ano
      WHEN 2024 THEN meta_alfabetizacao_2024
      WHEN 2025 THEN meta_alfabetizacao_2025
      WHEN 2026 THEN meta_alfabetizacao_2026
      WHEN 2027 THEN meta_alfabetizacao_2027
      WHEN 2028 THEN meta_alfabetizacao_2028
      WHEN 2029 THEN meta_alfabetizacao_2029
      WHEN 2030 THEN meta_alfabetizacao_2030
      ELSE NULL
    END AS meta_alfabetizacao_ano,

    taxa_observada_disponivel,
    participacao_disponivel,
    serie_metas_completa,
    status_serie_metas

  FROM
    `{PROJECT_ID}.{SILVER_DATASET}.meta_alfabetizacao_uf`
),

metricas AS (
  SELECT
    chave_ano_uf_rede,
    ano,
    sigla_uf,
    regiao,
    rede,
    resultado_alfabetizacao,
    percentual_participacao,

    ROUND(
      CAST(meta_alfabetizacao_ano AS NUMERIC),
      2
    ) AS meta_alfabetizacao_ano,

    taxa_observada_disponivel,
    participacao_disponivel,
    serie_metas_completa,
    status_serie_metas

  FROM
    metas_alinhadas
)

SELECT
  chave_ano_uf_rede,
  ano,
  sigla_uf,
  regiao,
  rede,
  resultado_alfabetizacao,
  meta_alfabetizacao_ano,

  CASE
    WHEN
      resultado_alfabetizacao IS NOT NULL
      AND meta_alfabetizacao_ano IS NOT NULL
    THEN ROUND(
      resultado_alfabetizacao
      - meta_alfabetizacao_ano,
      2
    )

    ELSE NULL
  END AS diferenca_resultado_meta,

  CASE
    WHEN
      resultado_alfabetizacao IS NOT NULL
      AND meta_alfabetizacao_ano IS NOT NULL
    THEN ROUND(
      GREATEST(
        meta_alfabetizacao_ano
        - resultado_alfabetizacao,
        0
      ),
      2
    )

    ELSE NULL
  END AS gap_para_meta,

  CASE
    WHEN
      resultado_alfabetizacao IS NOT NULL
      AND meta_alfabetizacao_ano IS NOT NULL
    THEN ROUND(
      GREATEST(
        resultado_alfabetizacao
        - meta_alfabetizacao_ano,
        0
      ),
      2
    )

    ELSE NULL
  END AS excedente_acima_meta,

  CASE
    WHEN resultado_alfabetizacao IS NULL
      THEN 'resultado_indisponivel'

    WHEN meta_alfabetizacao_ano IS NULL
      THEN 'meta_indisponivel_para_o_ano'

    WHEN resultado_alfabetizacao >= meta_alfabetizacao_ano
      THEN 'meta_atingida'

    ELSE 'meta_nao_atingida'
  END AS status_meta,

  percentual_participacao,
  taxa_observada_disponivel,
  participacao_disponivel,
  serie_metas_completa,
  status_serie_metas,
  CURRENT_TIMESTAMP() AS processado_em

FROM
  metricas;


CREATE OR REPLACE TABLE
  `{PROJECT_ID}.{GOLD_DATASET}.indicador_brasil`

AS

WITH metas_alinhadas AS (
  SELECT
    chave_ano_rede,
    ano,
    rede,

    ROUND(
      CAST(taxa_alfabetizacao_observada AS NUMERIC),
      2
    ) AS resultado_alfabetizacao,

    ROUND(
      CAST(percentual_participacao AS NUMERIC),
      2
    ) AS percentual_participacao,

    CASE ano
      WHEN 2024 THEN meta_alfabetizacao_2024
      WHEN 2025 THEN meta_alfabetizacao_2025
      WHEN 2026 THEN meta_alfabetizacao_2026
      WHEN 2027 THEN meta_alfabetizacao_2027
      WHEN 2028 THEN meta_alfabetizacao_2028
      WHEN 2029 THEN meta_alfabetizacao_2029
      WHEN 2030 THEN meta_alfabetizacao_2030
      ELSE NULL
    END AS meta_alfabetizacao_ano,

    taxa_observada_disponivel,
    participacao_disponivel,
    serie_metas_completa,
    status_serie_metas

  FROM
    `{PROJECT_ID}.{SILVER_DATASET}.meta_alfabetizacao_brasil`
),

metricas AS (
  SELECT
    chave_ano_rede,
    ano,
    rede,
    resultado_alfabetizacao,
    percentual_participacao,

    ROUND(
      CAST(meta_alfabetizacao_ano AS NUMERIC),
      2
    ) AS meta_alfabetizacao_ano,

    taxa_observada_disponivel,
    participacao_disponivel,
    serie_metas_completa,
    status_serie_metas

  FROM
    metas_alinhadas
)

SELECT
  chave_ano_rede,
  ano,
  rede,
  resultado_alfabetizacao,
  meta_alfabetizacao_ano,

  CASE
    WHEN
      resultado_alfabetizacao IS NOT NULL
      AND meta_alfabetizacao_ano IS NOT NULL
    THEN ROUND(
      resultado_alfabetizacao
      - meta_alfabetizacao_ano,
      2
    )

    ELSE NULL
  END AS diferenca_resultado_meta,

  CASE
    WHEN
      resultado_alfabetizacao IS NOT NULL
      AND meta_alfabetizacao_ano IS NOT NULL
    THEN ROUND(
      GREATEST(
        meta_alfabetizacao_ano
        - resultado_alfabetizacao,
        0
      ),
      2
    )

    ELSE NULL
  END AS gap_para_meta,

  CASE
    WHEN
      resultado_alfabetizacao IS NOT NULL
      AND meta_alfabetizacao_ano IS NOT NULL
    THEN ROUND(
      GREATEST(
        resultado_alfabetizacao
        - meta_alfabetizacao_ano,
        0
      ),
      2
    )

    ELSE NULL
  END AS excedente_acima_meta,

  CASE
    WHEN resultado_alfabetizacao IS NULL
      THEN 'resultado_indisponivel'

    WHEN meta_alfabetizacao_ano IS NULL
      THEN 'meta_indisponivel_para_o_ano'

    WHEN resultado_alfabetizacao >= meta_alfabetizacao_ano
      THEN 'meta_atingida'

    ELSE 'meta_nao_atingida'
  END AS status_meta,

  percentual_participacao,
  taxa_observada_disponivel,
  participacao_disponivel,
  serie_metas_completa,
  status_serie_metas,
  CURRENT_TIMESTAMP() AS processado_em

FROM
  metricas
"""

executar_sql(
    camada="GOLD",
    etapa="criar_gold_indicadores_uf_brasil",
    sql=sql_gold_indicadores_uf_brasil,
    executar=EXECUTE_PIPELINE,
)

display(
    pd.DataFrame(pipeline_logs).tail(1)
)

[PREPARADO] GOLD | criar_gold_indicadores_uf_brasil


,execution_id,camada,etapa,inicio_utc,fim_utc,duracao_segundos,status,bytes_processados,mensagem_erro
12,edd46773-51b2-4145-b489-09a398e33c9a,GOLD,criar_gold_indicadores_uf_brasil,2026-07-12T15:42:46.611560+00:00,2026-07-12T15:42:46.611581+00:00,0,PREPARADO,0,None


## 12. Evolução municipal e resumo regional da camada Gold

Esta etapa prepara duas tabelas analíticas.

### `gold.evolucao_municipio`

Compara o resultado de alfabetização de cada município entre 2023 e 2024.

A tabela apresenta:

- resultado de 2023;
- resultado de 2024;
- variação em pontos percentuais;
- classificação da evolução;
- meta e situação do município em 2024;
- informações territoriais.

A evolução será classificada como:

- `avancou`: aumento superior ou igual a 0,01 ponto percentual;
- `recuou`: redução superior ou igual a 0,01 ponto percentual;
- `estavel`: variação entre -0,01 e 0,01;
- `sem_base_2023`: resultado de 2023 indisponível;
- `sem_resultado_2024`: resultado de 2024 indisponível.

A variação será arredondada antes da classificação, evitando erros causados pela precisão de números de ponto flutuante.

### `gold.resumo_regiao`

Produz uma visão executiva por ano, região e rede, contendo:

- quantidade total de municípios;
- municípios com resultado e meta disponíveis;
- municípios que atingiram ou não atingiram a meta;
- médias de resultado, meta, diferença e gap;
- percentual de municípios que atingiram a meta.

As médias regionais representam médias simples entre municípios e não devem ser interpretadas como taxas oficiais ponderadas por população ou quantidade de alunos.

Como o modo seguro permanece ativo, as transformações serão apenas registradas como preparadas.

In [14]:
sql_gold_evolucao_resumo = f"""
CREATE OR REPLACE TABLE
  `{PROJECT_ID}.{GOLD_DATASET}.evolucao_municipio`

CLUSTER BY
  sigla_uf,
  id_municipio,
  rede

AS

WITH consolidacao AS (
  SELECT
    id_municipio,
    ANY_VALUE(nome_municipio) AS nome_municipio,
    ANY_VALUE(sigla_uf) AS sigla_uf,
    ANY_VALUE(regiao) AS regiao,
    rede,

    MAX(
      IF(
        ano = 2023,
        resultado_alfabetizacao,
        NULL
      )
    ) AS resultado_2023,

    MAX(
      IF(
        ano = 2024,
        resultado_alfabetizacao,
        NULL
      )
    ) AS resultado_2024,

    MAX(
      IF(
        ano = 2024,
        meta_alfabetizacao_ano,
        NULL
      )
    ) AS meta_2024,

    MAX(
      IF(
        ano = 2024,
        status_meta,
        NULL
      )
    ) AS status_meta_2024,

    MAX(
      IF(
        ano = 2024,
        percentual_participacao,
        NULL
      )
    ) AS percentual_participacao_2024

  FROM
    `{PROJECT_ID}.{GOLD_DATASET}.indicador_municipio`

  WHERE
    ano IN (2023, 2024)

  GROUP BY
    id_municipio,
    rede
),

metricas AS (
  SELECT
    id_municipio,
    nome_municipio,
    sigla_uf,
    regiao,
    rede,
    resultado_2023,
    resultado_2024,
    meta_2024,
    status_meta_2024,
    percentual_participacao_2024,

    CASE
      WHEN
        resultado_2023 IS NOT NULL
        AND resultado_2024 IS NOT NULL
      THEN ROUND(
        resultado_2024 - resultado_2023,
        2
      )

      ELSE NULL
    END AS variacao_pontos_percentuais

  FROM
    consolidacao
)

SELECT
  CONCAT(
    id_municipio,
    '|',
    rede
  ) AS chave_municipio_rede,

  id_municipio,
  nome_municipio,
  sigla_uf,
  regiao,
  rede,
  resultado_2023,
  resultado_2024,
  variacao_pontos_percentuais,

  CASE
    WHEN resultado_2023 IS NULL
      AND resultado_2024 IS NULL
      THEN 'sem_resultados'

    WHEN resultado_2023 IS NULL
      THEN 'sem_base_2023'

    WHEN resultado_2024 IS NULL
      THEN 'sem_resultado_2024'

    WHEN variacao_pontos_percentuais >= 0.01
      THEN 'avancou'

    WHEN variacao_pontos_percentuais <= -0.01
      THEN 'recuou'

    ELSE 'estavel'
  END AS status_evolucao,

  meta_2024,
  status_meta_2024,
  percentual_participacao_2024,
  CURRENT_TIMESTAMP() AS processado_em

FROM
  metricas;


CREATE OR REPLACE TABLE
  `{PROJECT_ID}.{GOLD_DATASET}.resumo_regiao`

CLUSTER BY
  regiao,
  rede

AS

WITH agregacao AS (
  SELECT
    ano,
    regiao,
    rede,

    COUNT(*) AS municipios_total,

    COUNTIF(
      resultado_alfabetizacao IS NOT NULL
    ) AS municipios_com_resultado,

    COUNTIF(
      meta_alfabetizacao_ano IS NOT NULL
    ) AS municipios_com_meta,

    COUNTIF(
      resultado_alfabetizacao IS NOT NULL
      AND meta_alfabetizacao_ano IS NOT NULL
    ) AS municipios_com_comparacao,

    COUNTIF(
      status_meta = 'meta_atingida'
    ) AS municipios_meta_atingida,

    COUNTIF(
      status_meta = 'meta_nao_atingida'
    ) AS municipios_meta_nao_atingida,

    ROUND(
      AVG(resultado_alfabetizacao),
      2
    ) AS media_resultado_alfabetizacao,

    ROUND(
      AVG(meta_alfabetizacao_ano),
      2
    ) AS media_meta_alfabetizacao,

    ROUND(
      AVG(diferenca_resultado_meta),
      2
    ) AS media_diferenca_resultado_meta,

    ROUND(
      AVG(gap_para_meta),
      2
    ) AS media_gap_para_meta

  FROM
    `{PROJECT_ID}.{GOLD_DATASET}.indicador_municipio`

  GROUP BY
    ano,
    regiao,
    rede
)

SELECT
  CONCAT(
    CAST(ano AS STRING),
    '|',
    regiao,
    '|',
    rede
  ) AS chave_ano_regiao_rede,

  ano,
  regiao,
  rede,
  municipios_total,
  municipios_com_resultado,
  municipios_com_meta,
  municipios_com_comparacao,
  municipios_meta_atingida,
  municipios_meta_nao_atingida,
  media_resultado_alfabetizacao,
  media_meta_alfabetizacao,
  media_diferenca_resultado_meta,
  media_gap_para_meta,

  ROUND(
    SAFE_DIVIDE(
      municipios_meta_atingida,
      municipios_com_comparacao
    ) * 100,
    2
  ) AS percentual_municipios_meta_atingida,

  CURRENT_TIMESTAMP() AS processado_em

FROM
  agregacao
"""

executar_sql(
    camada="GOLD",
    etapa="criar_gold_evolucao_resumo",
    sql=sql_gold_evolucao_resumo,
    executar=EXECUTE_PIPELINE,
)

display(
    pd.DataFrame(pipeline_logs).tail(1)
)

[PREPARADO] GOLD | criar_gold_evolucao_resumo


,execution_id,camada,etapa,inicio_utc,fim_utc,duracao_segundos,status,bytes_processados,mensagem_erro
13,edd46773-51b2-4145-b489-09a398e33c9a,GOLD,criar_gold_evolucao_resumo,2026-07-12T15:44:38.121932+00:00,2026-07-12T15:44:38.121951+00:00,0,PREPARADO,0,None


## 13. Base analítica por aluno e preparação para Machine Learning

Esta etapa prepara duas estruturas na camada Gold.

### `gold.aluno_analitico`

A tabela mantém todos os registros da Silver e acrescenta as informações territoriais da dimensão municipal:

- nome do município;
- UF;
- região;
- indicador de correspondência territorial.

O relacionamento será realizado somente pelo código oficial do município.

Essa decisão evita a perda de alunos que poderia ocorrer caso o enriquecimento dependesse simultaneamente de ano, série e rede.

A tabela preservará:

- avaliações válidas e inválidas;
- ausências;
- valores nulos de proficiência e peso;
- campos de auditoria da classificação pelo corte de 743 pontos.

Ela será particionada por ano e clusterizada por município e rede.

### `gold.base_ml_aluno`

A view disponibilizará somente avaliações válidas para uma futura etapa de Machine Learning.

A variável-alvo será definida explicitamente como `target_alfabetizado`, derivada da classificação oficial presente na fonte.

Para impedir vazamento de informação, a view não incluirá:

- proficiência;
- classificação calculada pelo corte de 743 pontos;
- indicador de coerência com o corte de 743 pontos.

Esses campos explicariam diretamente a variável-alvo e tornariam qualquer avaliação do modelo artificialmente otimista.

Os identificadores serão mantidos para rastreabilidade, mas não deverão ser utilizados como variáveis preditoras.

Também será criada uma divisão determinística:

- 70% para treino;
- 15% para validação;
- 15% para teste.

A criação da view não representa o treinamento de um modelo. Ela apenas prepara uma base segura e reproduzível para a próxima fase do projeto.

Como o modo seguro permanece ativo, as estruturas serão apenas registradas como preparadas.

In [15]:
sql_gold_aluno_ml = f"""
CREATE OR REPLACE TABLE
  `{PROJECT_ID}.{GOLD_DATASET}.aluno_analitico`

PARTITION BY
  RANGE_BUCKET(ano, GENERATE_ARRAY(2023, 2025, 1))

CLUSTER BY
  id_municipio,
  rede

AS

SELECT
  a.chave_ano_aluno,
  a.ano,
  a.id_municipio,
  d.nome_municipio,
  d.sigla_uf,
  d.regiao,
  a.id_escola,
  a.id_aluno,
  a.caderno,
  a.serie,
  a.rede,
  a.presenca,
  a.preenchimento_caderno,
  a.alfabetizado_oficial,
  a.proficiencia,
  a.peso_aluno,
  a.presente,
  a.caderno_preenchido,
  a.avaliacao_valida,
  a.status_avaliacao,
  a.alfabetizado_calculado_743,
  a.classificacao_coerente_743,
  d.id_municipio IS NOT NULL
    AS municipio_encontrado_na_dimensao,
  CURRENT_TIMESTAMP() AS processado_em

FROM
  `{PROJECT_ID}.{SILVER_DATASET}.alunos` AS a

LEFT JOIN
  `{PROJECT_ID}.{SILVER_DATASET}.dim_municipio` AS d
ON
  a.id_municipio = d.id_municipio;


CREATE OR REPLACE VIEW
  `{PROJECT_ID}.{GOLD_DATASET}.base_ml_aluno`

AS

WITH base AS (
  SELECT
    chave_ano_aluno,
    ano,
    id_municipio,
    nome_municipio,
    sigla_uf,
    regiao,
    id_escola,
    id_aluno,
    caderno,
    serie,
    rede,
    presenca,
    preenchimento_caderno,
    presente,
    caderno_preenchido,
    peso_aluno,
    alfabetizado_oficial
      AS target_alfabetizado,
    municipio_encontrado_na_dimensao,

    MOD(
      MOD(
        FARM_FINGERPRINT(chave_ano_aluno),
        100
      ) + 100,
      100
    ) AS numero_divisao

  FROM
    `{PROJECT_ID}.{GOLD_DATASET}.aluno_analitico`

  WHERE
    avaliacao_valida
    AND alfabetizado_oficial IS NOT NULL
)

SELECT
  chave_ano_aluno,
  ano,
  id_municipio,
  nome_municipio,
  sigla_uf,
  regiao,
  id_escola,
  id_aluno,
  caderno,
  serie,
  rede,
  presenca,
  preenchimento_caderno,
  presente,
  caderno_preenchido,
  peso_aluno,
  target_alfabetizado,
  municipio_encontrado_na_dimensao,

  CASE
    WHEN numero_divisao < 70
      THEN 'treino'

    WHEN numero_divisao < 85
      THEN 'validacao'

    ELSE 'teste'
  END AS divisao_ml

FROM
  base
"""

executar_sql(
    camada="GOLD",
    etapa="criar_gold_aluno_e_base_ml",
    sql=sql_gold_aluno_ml,
    executar=EXECUTE_PIPELINE,
)

display(
    pd.DataFrame(pipeline_logs).tail(1)
)

[PREPARADO] GOLD | criar_gold_aluno_e_base_ml


,execution_id,camada,etapa,inicio_utc,fim_utc,duracao_segundos,status,bytes_processados,mensagem_erro
14,edd46773-51b2-4145-b489-09a398e33c9a,GOLD,criar_gold_aluno_e_base_ml,2026-07-12T15:46:50.330669+00:00,2026-07-12T15:46:50.330686+00:00,0,PREPARADO,0,None


## 14. Quality gate consolidado da camada Gold

Esta etapa valida as estruturas analíticas finais da pipeline.

Serão verificados:

- volume esperado de registros;
- unicidade das chaves analíticas;
- consistência dos cálculos de resultado, meta, diferença, gap e excedente;
- coerência das classificações de cumprimento das metas;
- consistência da evolução municipal;
- integridade dos indicadores regionais;
- correspondência territorial dos alunos;
- volume da base destinada a Machine Learning;
- validade das divisões de treino, validação e teste;
- ausência de variáveis que causariam vazamento de informação na view de Machine Learning.

A validação é executada somente em modo de leitura.

In [18]:
sql_validar_gold = f"""
WITH

indicadores AS (
  SELECT
    'indicador_municipio' AS tabela,
    chave_ano_municipio_rede AS chave,
    resultado_alfabetizacao AS resultado,
    meta_alfabetizacao_ano AS meta,
    diferenca_resultado_meta AS diferenca,
    gap_para_meta AS gap,
    excedente_acima_meta AS excedente,
    status_meta AS status

  FROM
    `{PROJECT_ID}.{GOLD_DATASET}.indicador_municipio`

  UNION ALL

  SELECT
    'indicador_uf' AS tabela,
    chave_ano_uf_rede AS chave,
    resultado_alfabetizacao AS resultado,
    meta_alfabetizacao_ano AS meta,
    diferenca_resultado_meta AS diferenca,
    gap_para_meta AS gap,
    excedente_acima_meta AS excedente,
    status_meta AS status

  FROM
    `{PROJECT_ID}.{GOLD_DATASET}.indicador_uf`

  UNION ALL

  SELECT
    'indicador_brasil' AS tabela,
    chave_ano_rede AS chave,
    resultado_alfabetizacao AS resultado,
    meta_alfabetizacao_ano AS meta,
    diferenca_resultado_meta AS diferenca,
    gap_para_meta AS gap,
    excedente_acima_meta AS excedente,
    status_meta AS status

  FROM
    `{PROJECT_ID}.{GOLD_DATASET}.indicador_brasil`
),

indicadores_esperados AS (
  SELECT
    'indicador_municipio' AS tabela,
    COUNT(*) AS linhas_esperadas

  FROM
    `{PROJECT_ID}.{SILVER_DATASET}.meta_alfabetizacao_municipio`

  UNION ALL

  SELECT
    'indicador_uf' AS tabela,
    COUNT(*) AS linhas_esperadas

  FROM
    `{PROJECT_ID}.{SILVER_DATASET}.meta_alfabetizacao_uf`

  UNION ALL

  SELECT
    'indicador_brasil' AS tabela,
    COUNT(*) AS linhas_esperadas

  FROM
    `{PROJECT_ID}.{SILVER_DATASET}.meta_alfabetizacao_brasil`
),

indicadores_validacao AS (
  SELECT
    tabela,
    COUNT(*) AS linhas_obtidas,
    COUNT(DISTINCT chave) AS chaves_distintas,

    COUNTIF(
      status IS DISTINCT FROM
        CASE
          WHEN resultado IS NULL
            THEN 'resultado_indisponivel'

          WHEN meta IS NULL
            THEN 'meta_indisponivel_para_o_ano'

          WHEN resultado >= meta
            THEN 'meta_atingida'

          ELSE 'meta_nao_atingida'
        END

      OR diferenca IS DISTINCT FROM
        CASE
          WHEN
            resultado IS NOT NULL
            AND meta IS NOT NULL
          THEN ROUND(
            resultado - meta,
            2
          )

          ELSE NULL
        END

      OR gap IS DISTINCT FROM
        CASE
          WHEN
            resultado IS NOT NULL
            AND meta IS NOT NULL
          THEN ROUND(
            GREATEST(
              meta - resultado,
              0
            ),
            2
          )

          ELSE NULL
        END

      OR excedente IS DISTINCT FROM
        CASE
          WHEN
            resultado IS NOT NULL
            AND meta IS NOT NULL
          THEN ROUND(
            GREATEST(
              resultado - meta,
              0
            ),
            2
          )

          ELSE NULL
        END
    ) AS problemas_integridade

  FROM
    indicadores

  GROUP BY
    tabela
),

evolucao_esperada AS (
  SELECT
    COUNT(*) AS linhas_esperadas

  FROM (
    SELECT DISTINCT
      id_municipio,
      rede

    FROM
      `{PROJECT_ID}.{GOLD_DATASET}.indicador_municipio`

    WHERE
      ano IN (2023, 2024)
  )
),

evolucao_validacao AS (
  SELECT
    COUNT(*) AS linhas_obtidas,
    COUNT(DISTINCT chave_municipio_rede)
      AS chaves_distintas,

    COUNTIF(
      variacao_pontos_percentuais IS DISTINCT FROM
        CASE
          WHEN
            resultado_2023 IS NOT NULL
            AND resultado_2024 IS NOT NULL
          THEN ROUND(
            resultado_2024 - resultado_2023,
            2
          )

          ELSE NULL
        END

      OR status_evolucao IS DISTINCT FROM
        CASE
          WHEN
            resultado_2023 IS NULL
            AND resultado_2024 IS NULL
          THEN 'sem_resultados'

          WHEN resultado_2023 IS NULL
            THEN 'sem_base_2023'

          WHEN resultado_2024 IS NULL
            THEN 'sem_resultado_2024'

          WHEN ROUND(
            resultado_2024 - resultado_2023,
            2
          ) >= 0.01
            THEN 'avancou'

          WHEN ROUND(
            resultado_2024 - resultado_2023,
            2
          ) <= -0.01
            THEN 'recuou'

          ELSE 'estavel'
        END
    ) AS problemas_integridade

  FROM
    `{PROJECT_ID}.{GOLD_DATASET}.evolucao_municipio`
),

resumo_esperado AS (
  SELECT
    COUNT(*) AS linhas_esperadas

  FROM (
    SELECT DISTINCT
      ano,
      regiao,
      rede

    FROM
      `{PROJECT_ID}.{GOLD_DATASET}.indicador_municipio`
  )
),

resumo_validacao AS (
  SELECT
    COUNT(*) AS linhas_obtidas,
    COUNT(DISTINCT chave_ano_regiao_rede)
      AS chaves_distintas,

    COUNTIF(
      municipios_com_comparacao
        != municipios_meta_atingida
        + municipios_meta_nao_atingida

      OR municipios_total
        < municipios_com_resultado

      OR municipios_total
        < municipios_com_meta

      OR percentual_municipios_meta_atingida
        IS DISTINCT FROM
          ROUND(
            SAFE_DIVIDE(
              municipios_meta_atingida,
              municipios_com_comparacao
            ) * 100,
            2
          )
    ) AS problemas_integridade

  FROM
    `{PROJECT_ID}.{GOLD_DATASET}.resumo_regiao`
),

alunos_esperados AS (
  SELECT
    COUNT(*) AS linhas_esperadas

  FROM
    `{PROJECT_ID}.{SILVER_DATASET}.alunos`
),

alunos_validacao AS (
  SELECT
    COUNT(*) AS linhas_obtidas,
    COUNT(DISTINCT chave_ano_aluno)
      AS chaves_distintas,

    COUNTIF(
      NOT COALESCE(
        municipio_encontrado_na_dimensao,
        FALSE
      )

      OR nome_municipio IS NULL
      OR sigla_uf IS NULL
      OR regiao IS NULL
    ) AS problemas_integridade

  FROM
    `{PROJECT_ID}.{GOLD_DATASET}.aluno_analitico`
),

ml_esperado AS (
  SELECT
    COUNTIF(
      avaliacao_valida
      AND alfabetizado_oficial IS NOT NULL
    ) AS linhas_esperadas

  FROM
    `{PROJECT_ID}.{SILVER_DATASET}.alunos`
),

ml_validacao AS (
  SELECT
    COUNT(*) AS linhas_obtidas,
    COUNT(DISTINCT chave_ano_aluno)
      AS chaves_distintas,

    COUNTIF(
      target_alfabetizado IS NULL

      OR target_alfabetizado NOT IN (0, 1)

      OR divisao_ml IS NULL

      OR divisao_ml NOT IN (
        'treino',
        'validacao',
        'teste'
      )

      OR NOT COALESCE(
        municipio_encontrado_na_dimensao,
        FALSE
      )
    ) AS problemas_integridade

  FROM
    `{PROJECT_ID}.{GOLD_DATASET}.base_ml_aluno`
),

ml_colunas AS (
  SELECT
    COUNTIF(
      column_name IN (
        'proficiencia',
        'alfabetizado_calculado_743',
        'classificacao_coerente_743'
      )
    ) AS colunas_com_vazamento

  FROM
    `{PROJECT_ID}.{GOLD_DATASET}.INFORMATION_SCHEMA.COLUMNS`

  WHERE
    table_name = 'base_ml_aluno'
)

SELECT
  v.tabela,
  e.linhas_esperadas,
  v.linhas_obtidas,

  e.linhas_esperadas = v.linhas_obtidas
    AS contagem_confere,

  v.chaves_distintas,

  v.linhas_obtidas = v.chaves_distintas
    AS chave_unica_confere,

  v.problemas_integridade,

  v.problemas_integridade = 0
    AS validacao_especifica

FROM
  indicadores_validacao AS v

JOIN
  indicadores_esperados AS e
USING
  (tabela)

UNION ALL

SELECT
  'evolucao_municipio' AS tabela,
  e.linhas_esperadas,
  v.linhas_obtidas,
  e.linhas_esperadas = v.linhas_obtidas,
  v.chaves_distintas,
  v.linhas_obtidas = v.chaves_distintas,
  v.problemas_integridade,
  v.problemas_integridade = 0

FROM
  evolucao_esperada AS e

CROSS JOIN
  evolucao_validacao AS v

UNION ALL

SELECT
  'resumo_regiao' AS tabela,
  e.linhas_esperadas,
  v.linhas_obtidas,
  e.linhas_esperadas = v.linhas_obtidas,
  v.chaves_distintas,
  v.linhas_obtidas = v.chaves_distintas,
  v.problemas_integridade,
  v.problemas_integridade = 0

FROM
  resumo_esperado AS e

CROSS JOIN
  resumo_validacao AS v

UNION ALL

SELECT
  'aluno_analitico' AS tabela,
  e.linhas_esperadas,
  v.linhas_obtidas,
  e.linhas_esperadas = v.linhas_obtidas,
  v.chaves_distintas,
  v.linhas_obtidas = v.chaves_distintas,
  v.problemas_integridade,
  v.problemas_integridade = 0

FROM
  alunos_esperados AS e

CROSS JOIN
  alunos_validacao AS v

UNION ALL

SELECT
  'base_ml_aluno' AS tabela,
  e.linhas_esperadas,
  v.linhas_obtidas,
  e.linhas_esperadas = v.linhas_obtidas,
  v.chaves_distintas,
  v.linhas_obtidas = v.chaves_distintas,
  v.problemas_integridade,

  c.colunas_com_vazamento = 0
    AND v.problemas_integridade = 0
    AS validacao_especifica

FROM
  ml_esperado AS e

CROSS JOIN
  ml_validacao AS v

CROSS JOIN
  ml_colunas AS c

ORDER BY
  tabela
"""

print("Quality gate Gold preparado.")
print(
    "A validação será executada após "
    "a reconstrução da camada Gold."
)

Quality gate Gold preparado.
A validação será executada após a reconstrução da camada Gold.


## 15. Execução real da pipeline end-to-end

Após a preparação e a validação individual dos scripts, a execução completa será liberada.

O fluxo seguirá obrigatoriamente esta ordem:

1. Atualização da Bronze;
2. Quality gate da Bronze;
3. Construção da Silver;
4. Quality gate da Silver;
5. Construção da Gold;
6. Quality gate da Gold;
7. Consolidação dos logs.

O pipeline será interrompido imediatamente caso qualquer quality gate seja reprovado.

In [20]:
EXECUTE_PIPELINE = True

pipeline_logs.clear()

pipeline_started_at = datetime.now(timezone.utc)

print("=" * 70)
print("INÍCIO DA EXECUÇÃO END-TO-END")
print("=" * 70)

for step in bronze_steps:
    executar_sql(
        camada=step["camada"],
        etapa=step["etapa"],
        sql=step["sql"],
        executar=EXECUTE_PIPELINE,
    )

bronze_validation_df, bronze_quality_gate = validar_bronze()

display(bronze_validation_df)

print(
    "\nQuality gate Bronze:",
    "APROVADO" if bronze_quality_gate else "REPROVADO",
)

if not bronze_quality_gate:
    raise RuntimeError(
        "Pipeline interrompido no quality gate Bronze."
    )

silver_steps = [
    (
        "criar_silver_alunos",
        sql_silver_alunos,
    ),
    (
        "criar_silver_territorio",
        sql_silver_territorio,
    ),
    (
        "criar_silver_metas",
        sql_silver_metas,
    ),
]

for etapa, sql in silver_steps:
    executar_sql(
        camada="SILVER",
        etapa=etapa,
        sql=sql,
        executar=EXECUTE_PIPELINE,
    )

silver_validation_df = (
    client
    .query(
        sql_validar_silver,
        location=LOCATION,
    )
    .result()
    .to_dataframe()
)

display(silver_validation_df)

silver_quality_gate = bool(
    silver_validation_df["contagem_confere"].all()
    and silver_validation_df["chave_unica_confere"].all()
    and (
        silver_validation_df["problemas_integridade"] == 0
    ).all()
    and silver_validation_df["ausencias_preservadas"].all()
)

print(
    "\nQuality gate Silver:",
    "APROVADO" if silver_quality_gate else "REPROVADO",
)

if not silver_quality_gate:
    raise RuntimeError(
        "Pipeline interrompido no quality gate Silver."
    )

sql_remover_gold_anterior = f"""
DROP VIEW IF EXISTS
  `{PROJECT_ID}.{GOLD_DATASET}.base_ml_aluno`;

DROP TABLE IF EXISTS
  `{PROJECT_ID}.{GOLD_DATASET}.aluno_analitico`;

DROP TABLE IF EXISTS
  `{PROJECT_ID}.{GOLD_DATASET}.resumo_regiao`;

DROP TABLE IF EXISTS
  `{PROJECT_ID}.{GOLD_DATASET}.evolucao_municipio`;

DROP TABLE IF EXISTS
  `{PROJECT_ID}.{GOLD_DATASET}.indicador_brasil`;

DROP TABLE IF EXISTS
  `{PROJECT_ID}.{GOLD_DATASET}.indicador_uf`;

DROP TABLE IF EXISTS
  `{PROJECT_ID}.{GOLD_DATASET}.indicador_municipio`;
"""

executar_sql(
    camada="GOLD",
    etapa="remover_estruturas_gold_anteriores",
    sql=sql_remover_gold_anterior,
    executar=EXECUTE_PIPELINE,
)

gold_steps = [
    (
        "criar_gold_indicador_municipio",
        sql_gold_indicador_municipio,
    ),
    (
        "criar_gold_indicadores_uf_brasil",
        sql_gold_indicadores_uf_brasil,
    ),
    (
        "criar_gold_evolucao_resumo",
        sql_gold_evolucao_resumo,
    ),
    (
        "criar_gold_aluno_e_base_ml",
        sql_gold_aluno_ml,
    ),
]

for etapa, sql in gold_steps:
    executar_sql(
        camada="GOLD",
        etapa=etapa,
        sql=sql,
        executar=EXECUTE_PIPELINE,
    )

gold_validation_df = (
    client
    .query(
        sql_validar_gold,
        location=LOCATION,
    )
    .result()
    .to_dataframe()
)

display(gold_validation_df)

gold_quality_gate = bool(
    gold_validation_df["contagem_confere"].all()
    and gold_validation_df["chave_unica_confere"].all()
    and (
        gold_validation_df["problemas_integridade"] == 0
    ).all()
    and gold_validation_df["validacao_especifica"].all()
)

print(
    "\nQuality gate Gold:",
    "APROVADO" if gold_quality_gate else "REPROVADO",
)

if not gold_quality_gate:
    raise RuntimeError(
        "Pipeline interrompido no quality gate Gold."
    )

pipeline_ended_at = datetime.now(timezone.utc)

pipeline_logs_df = pd.DataFrame(pipeline_logs)

display(pipeline_logs_df)

print("\n" + "=" * 70)
print("PIPELINE END-TO-END CONCLUÍDO COM SUCESSO")
print("=" * 70)
print(f"Execution ID: {execution_id}")
print(f"Início UTC: {pipeline_started_at.isoformat()}")
print(f"Fim UTC: {pipeline_ended_at.isoformat()}")

print(
    "Duração total:",
    round(
        (
            pipeline_ended_at
            - pipeline_started_at
        ).total_seconds(),
        2,
    ),
    "segundos",
)

print(
    "Bytes processados:",
    int(
        pipeline_logs_df[
            "bytes_processados"
        ].fillna(0).sum()
    ),
)

INÍCIO DA EXECUÇÃO END-TO-END
[SUCESSO] BRONZE | carregar_alunos | 4.95s
[SUCESSO] BRONZE | carregar_dicionario | 1.92s
[SUCESSO] BRONZE | carregar_meta_alfabetizacao_brasil | 2.02s
[SUCESSO] BRONZE | carregar_meta_alfabetizacao_municipio | 2.62s
[SUCESSO] BRONZE | carregar_meta_alfabetizacao_uf | 1.95s
[SUCESSO] BRONZE | carregar_municipio | 1.95s
[SUCESSO] BRONZE | carregar_uf | 1.85s


,tabela,origem_existe,bronze_existe,linhas_origem,linhas_bronze,contagem_confere,mb_origem,mb_bronze
0,alunos,True,True,3867999,3867999,True,256.10,256.10
1,dicionario,True,True,27,27,True,0.00,0.00
2,meta_alfabetizacao_brasil,True,True,3,3,True,0.00,0.00
3,meta_alfabetizacao_municipio,True,True,10704,10704,True,1.10,1.10
4,meta_alfabetizacao_uf,True,True,81,81,True,0.01,0.01
5,municipio,True,True,23995,23995,True,1.75,1.75
6,uf,True,True,145,145,True,0.01,0.01



Quality gate Bronze: APROVADO
[SUCESSO] SILVER | criar_silver_alunos | 6.3s
[SUCESSO] SILVER | criar_silver_territorio | 21.37s
[SUCESSO] SILVER | criar_silver_metas | 7.64s


,tabela,linhas_fonte,linhas_silver,contagem_confere,chaves_distintas,chave_unica_confere,problemas_integridade,ausencias_preservadas
0,alunos,3867999,3867999,True,3867999,True,0,True
1,dim_municipio,5571,5571,True,5571,True,0,True
2,meta_alfabetizacao_brasil,3,3,True,3,True,0,True
3,meta_alfabetizacao_municipio,10704,10704,True,10704,True,0,True
4,meta_alfabetizacao_uf,81,81,True,81,True,0,True
5,municipio,23995,23995,True,23995,True,0,True
6,uf,145,145,True,145,True,0,True



Quality gate Silver: APROVADO
[SUCESSO] GOLD | remover_estruturas_gold_anteriores | 4.32s
[SUCESSO] GOLD | criar_gold_indicador_municipio | 2.1s
[SUCESSO] GOLD | criar_gold_indicadores_uf_brasil | 3.84s
[SUCESSO] GOLD | criar_gold_evolucao_resumo | 4.25s
[SUCESSO] GOLD | criar_gold_aluno_e_base_ml | 4.38s


,tabela,linhas_esperadas,linhas_obtidas,contagem_confere,chaves_distintas,chave_unica_confere,problemas_integridade,validacao_especifica
0,aluno_analitico,3867999,3867999,True,3867999,True,0,True
1,base_ml_aluno,3354661,3354661,True,3354661,True,0,True
2,evolucao_municipio,5352,5352,True,5352,True,0,True
3,indicador_brasil,3,3,True,3,True,0,True
4,indicador_municipio,10704,10704,True,10704,True,0,True
5,indicador_uf,81,81,True,81,True,0,True
6,resumo_regiao,10,10,True,10,True,0,True



Quality gate Gold: APROVADO


,execution_id,camada,etapa,inicio_utc,fim_utc,duracao_segundos,status,bytes_processados,mensagem_erro
0,edd46773-51b2-4145-b489-09a398e33c9a,BRONZE,carregar_alunos,2026-07-12T15:58:05.150517+00:00,2026-07-12T15:58:10.103322+00:00,4.95,SUCESSO,268545240,None
1,edd46773-51b2-4145-b489-09a398e33c9a,BRONZE,carregar_dicionario,2026-07-12T15:58:10.103416+00:00,2026-07-12T15:58:12.018829+00:00,1.92,SUCESSO,1174,None
2,edd46773-51b2-4145-b489-09a398e33c9a,BRONZE,carregar_meta_alfabetizacao_brasil,2026-07-12T15:58:12.018933+00:00,2026-07-12T15:58:14.039152+00:00,2.02,SUCESSO,270,None
3,edd46773-51b2-4145-b489-09a398e33c9a,BRONZE,carregar_meta_alfabetizacao_municipio,2026-07-12T15:58:14.039249+00:00,2026-07-12T15:58:16.662637+00:00,2.62,SUCESSO,1151232,None
4,edd46773-51b2-4145-b489-09a398e33c9a,BRONZE,carregar_meta_alfabetizacao_uf,2026-07-12T15:58:16.662723+00:00,2026-07-12T15:58:18.609039+00:00,1.95,SUCESSO,7374,None
5,edd46773-51b2-4145-b489-09a398e33c9a,BRONZE,carregar_municipio,2026-07-12T15:58:18.609137+00:00,2026-07-12T15:58:20.564089+00:00,1.95,SUCESSO,1832061,None
6,edd46773-51b2-4145-b489-09a398e33c9a,BRONZE,carregar_uf,2026-07-12T15:58:20.564172+00:00,2026-07-12T15:58:22.414622+00:00,1.85,SUCESSO,10330,None
7,edd46773-51b2-4145-b489-09a398e33c9a,SILVER,criar_silver_alunos,2026-07-12T15:58:25.291677+00:00,2026-07-12T15:58:31.588319+00:00,6.30,SUCESSO,268545240,None
8,edd46773-51b2-4145-b489-09a398e33c9a,SILVER,criar_silver_territorio,2026-07-12T15:58:31.588410+00:00,2026-07-12T15:58:52.957561+00:00,21.37,SUCESSO,2144573,None
9,edd46773-51b2-4145-b489-09a398e33c9a,SILVER,criar_silver_metas,2026-07-12T15:58:52.957650+00:00,2026-07-12T15:59:00.599739+00:00,7.64,SUCESSO,1309967,None



PIPELINE END-TO-END CONCLUÍDO COM SUCESSO
Execution ID: edd46773-51b2-4145-b489-09a398e33c9a
Início UTC: 2026-07-12T15:58:05.150224+00:00
Fim UTC: 2026-07-12T15:59:29.859284+00:00
Duração total: 84.71 segundos
Bytes processados: 1039246040


## 16. Exportação dos artefatos finais

Esta etapa preserva no Google Drive as evidências da execução end-to-end:

- logs de todas as etapas;
- resultados dos quality gates;
- manifesto da execução;
- scripts SQL consolidados;
- script de reconstrução da camada Gold.

Os arquivos serão armazenados na pasta:

`tech_challenge_alfabetizacao/pipeline_end_to_end`

Esses artefatos poderão ser adicionados ao repositório GitHub e utilizados como evidência técnica no README e na apresentação.

In [21]:
from google.colab import drive

variaveis_obrigatorias = [
    "pipeline_logs_df",
    "pipeline_started_at",
    "pipeline_ended_at",
    "bronze_validation_df",
    "silver_validation_df",
    "gold_validation_df",
    "bronze_quality_gate",
    "silver_quality_gate",
    "gold_quality_gate",
    "sql_remover_gold_anterior",
]

variaveis_ausentes = [
    nome
    for nome in variaveis_obrigatorias
    if nome not in globals()
]

if variaveis_ausentes:
    raise RuntimeError(
        "A célula 15 precisa ser concluída antes da exportação. "
        f"Variáveis ausentes: {variaveis_ausentes}"
    )

if not (
    bronze_quality_gate
    and silver_quality_gate
    and gold_quality_gate
):
    raise RuntimeError(
        "A exportação foi interrompida porque pelo menos "
        "um quality gate não foi aprovado."
    )

drive.mount(
    "/content/drive",
    force_remount=False,
)

output_dir = Path(
    "/content/drive/MyDrive/"
    "tech_challenge_alfabetizacao/"
    "pipeline_end_to_end"
)

sql_dir = output_dir / "sql"
quality_dir = output_dir / "quality_gates"
logs_dir = output_dir / "logs"

output_dir.mkdir(
    parents=True,
    exist_ok=True,
)

sql_dir.mkdir(
    parents=True,
    exist_ok=True,
)

quality_dir.mkdir(
    parents=True,
    exist_ok=True,
)

logs_dir.mkdir(
    parents=True,
    exist_ok=True,
)

pipeline_logs_df.to_csv(
    logs_dir / "pipeline_logs.csv",
    index=False,
)

pipeline_logs_df.to_json(
    logs_dir / "pipeline_logs.jsonl",
    orient="records",
    lines=True,
    force_ascii=False,
)

bronze_validation_df.to_csv(
    quality_dir / "quality_gate_bronze.csv",
    index=False,
)

silver_validation_df.to_csv(
    quality_dir / "quality_gate_silver.csv",
    index=False,
)

gold_validation_df.to_csv(
    quality_dir / "quality_gate_gold.csv",
    index=False,
)

bronze_sql = "\n\n".join(
    step["sql"].strip() + ";"
    for step in bronze_steps
)

sql_files = {
    "01_bronze_full_refresh.sql":
        bronze_sql,

    "02_silver_alunos.sql":
        sql_silver_alunos,

    "03_silver_territorio.sql":
        sql_silver_territorio,

    "04_silver_metas.sql":
        sql_silver_metas,

    "05_reconstruir_gold.sql":
        sql_remover_gold_anterior,

    "06_gold_indicador_municipio.sql":
        sql_gold_indicador_municipio,

    "07_gold_indicadores_uf_brasil.sql":
        sql_gold_indicadores_uf_brasil,

    "08_gold_evolucao_resumo.sql":
        sql_gold_evolucao_resumo,

    "09_gold_aluno_ml.sql":
        sql_gold_aluno_ml,

    "10_quality_gate_silver.sql":
        sql_validar_silver,

    "11_quality_gate_gold.sql":
        sql_validar_gold,
}

for filename, sql in sql_files.items():
    caminho = sql_dir / filename

    caminho.write_text(
        sql.strip(),
        encoding="utf-8",
    )

duracao_total = round(
    (
        pipeline_ended_at
        - pipeline_started_at
    ).total_seconds(),
    2,
)

bytes_processados = int(
    pipeline_logs_df[
        "bytes_processados"
    ].fillna(0).sum()
)

manifest = {
    "execution_id": execution_id,
    "project_id": PROJECT_ID,
    "location": LOCATION,
    "inicio_utc": pipeline_started_at.isoformat(),
    "fim_utc": pipeline_ended_at.isoformat(),
    "duracao_segundos": duracao_total,
    "quality_gate_bronze": bool(
        bronze_quality_gate
    ),
    "quality_gate_silver": bool(
        silver_quality_gate
    ),
    "quality_gate_gold": bool(
        gold_quality_gate
    ),
    "pipeline_concluida": bool(
        bronze_quality_gate
        and silver_quality_gate
        and gold_quality_gate
    ),
    "etapas_executadas": int(
        len(pipeline_logs_df)
    ),
    "etapas_com_sucesso": int(
        (
            pipeline_logs_df["status"]
            == "SUCESSO"
        ).sum()
    ),
    "bytes_processados": bytes_processados,
    "scripts_sql_exportados": len(
        sql_files
    ),
    "datasets": {
        "bronze": BRONZE_DATASET,
        "silver": SILVER_DATASET,
        "gold": GOLD_DATASET,
    },
}

with open(
    output_dir / "manifest.json",
    "w",
    encoding="utf-8",
) as arquivo:
    json.dump(
        manifest,
        arquivo,
        ensure_ascii=False,
        indent=2,
    )

resumo_execucao_df = pd.DataFrame(
    [
        {
            "execution_id": execution_id,
            "inicio_utc":
                pipeline_started_at.isoformat(),
            "fim_utc":
                pipeline_ended_at.isoformat(),
            "duracao_segundos":
                duracao_total,
            "etapas_executadas":
                len(pipeline_logs_df),
            "bytes_processados":
                bytes_processados,
            "bronze_aprovada":
                bronze_quality_gate,
            "silver_aprovada":
                silver_quality_gate,
            "gold_aprovada":
                gold_quality_gate,
        }
    ]
)

resumo_execucao_df.to_csv(
    output_dir / "resumo_execucao.csv",
    index=False,
)

print("=" * 70)
print("ARTEFATOS FINAIS EXPORTADOS")
print("=" * 70)
print(f"Diretório principal: {output_dir}")
print(f"Etapas registradas: {len(pipeline_logs_df)}")
print(f"Scripts SQL exportados: {len(sql_files)}")
print(f"Duração total: {duracao_total} segundos")
print(f"Bytes processados: {bytes_processados}")
print("Quality gate Bronze: APROVADO")
print("Quality gate Silver: APROVADO")
print("Quality gate Gold: APROVADO")
print("Pipeline end-to-end encerrada.")

Mounted at /content/drive
ARTEFATOS FINAIS EXPORTADOS
Diretório principal: /content/drive/MyDrive/tech_challenge_alfabetizacao/pipeline_end_to_end
Etapas registradas: 15
Scripts SQL exportados: 11
Duração total: 84.71 segundos
Bytes processados: 1039246040
Quality gate Bronze: APROVADO
Quality gate Silver: APROVADO
Quality gate Gold: APROVADO
Pipeline end-to-end encerrada.
